# Instruction-tuned stitching - does an IT recipient change what a stitch can carry?

**VRAM: 48 GB is enough** (donor gemma-2-9b/9b-it ~18 GB + recipient gemma-2-2b-it ~5 GB, bf16).

Two arms, switched by `ARM` in CELL 2 - run the notebook twice:
* `ARM = "it2it"`   - donor `gemma-2-9b-it` -> recipient `gemma-2-2b-it` (matched)
* `ARM = "base2it"` - donor `gemma-2-9b` (BASE) -> recipient `gemma-2-2b-it` (mismatched)

**The headline test (CELL 13b).** v1 showed conferral *coincides with leaving* the recipient's
natural manifold: the recon map (faithful translation, reconstruction cosine ~0.98) confers almost
nothing, while the task map lands off-manifold (cosine ~0.22) and confers ~0.89. If instruction
tuning keeps answer-related features closer to the recipient's natural manifold, the **recon map
should beat its 0.19 base-model floor**. CELL 13b sweeps the KL + MSE-anchor objective to trace the
manifold/conferral trade-off directly.

**v1 base-model reference numbers to compare against:**

| quantity | base gemma-2-2b recipient |
|---|---|
| recon conferral / recon cosine | 0.19 / 0.981 |
| task conferral / task cosine | 0.886 / 0.219 |
| KL sweep: cosine 0.24 -> 0.95 | conferral 0.945 -> 0.282 |

Run order: CELL 1 -> **restart kernel** -> run down. Arithmetic only (GSM8K / dose-response off by
default). Naming is legacy: `_9b` = DONOR, `_2b` = RECIPIENT.

In [1]:
# === CELL 1: install (run once, then RESTART KERNEL) ===
# torchvision/torchaudio are uninstalled BEFORE any torch import: a version mismatch between
# them and torch makes `import transformers` crash. sae_lens is installed HERE (not mid-run)
# so it cannot re-tug torch/transformers after a 27B donor is already sitting in VRAM.
!pip install -q "transformers==4.46.3" accelerate bitsandbytes numpy matplotlib datasets hf_transfer sae_lens
!pip uninstall -y torchvision torchaudio
# >>> RESTART THE KERNEL NOW, before running any other cell. <<<
# (Nothing above imports torch; everything below assumes a fresh kernel.)
# --- Blackwell/sm_120 pods ONLY (verify cell prints cap (12,0)): uncomment, run, restart again ---
# !pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu128


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
notebook 6.5.5 requires pyzmq<25,>=17, but you have pyzmq 26.0.0 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Found existing installation: torchvision 0.19.1+cu124
Uninstalling torchvision-0.19.1+cu124:
  Successfully uninstalled torchvision-0.19.1+cu124
Found existing installation: torchaudio 2.4.1+cu124
Uninstalling torchaudio-2.4.1+cu124:
  Successfully uninstalled torchaudio-2.4.1+cu124


In [10]:
import torch, transformers
print("torch:", torch.__version__, "| cuda cap:", torch.cuda.get_device_capability(0))
print("transformers:", transformers.__version__, "(want 4.46.3)")
free, total = torch.cuda.mem_get_info()
print(f"GPU: {torch.cuda.get_device_name(0)} | total {total/2**30:.1f} GiB | free {free/2**30:.1f} GiB")



torch: 2.4.1+cu124 | cuda cap: (8, 9)
transformers: 4.46.3 (want 4.46.3)
GPU: NVIDIA L40S | total 44.5 GiB | free 44.1 GiB


In [11]:
# === CELL 2: imports, set_submodule shim, global config (VRAM/compute switches) ===
# VERBATIM from CELL 2 of `Emergence_Gemma (4).ipynb` apart from the model ids, the layer
# START GUESSES, and the batch sizes (see the header markdown for why).
#
# NAMING: the `_2b` suffix means RECIPIENT and the `_9b` suffix means DONOR throughout. Those
# names are legacy (the validated v1 pair was 2B<-9B) and are kept so the shared helper cells
# stay byte-identical to the source notebooks. Here: DONOR = gemma-2-9b(-it), RECIPIENT = gemma-2-2b-it.
import os, json, math, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# shim: newer transformers' 4-bit path calls nn.Module.set_submodule, absent on older torch
if not hasattr(nn.Module, "set_submodule"):
    def _set_submodule(self, target, module):
        mod = self
        atoms = target.split(".")
        for a in atoms[:-1]:
            mod = getattr(mod, a)
        setattr(mod, atoms[-1], module)
    nn.Module.set_submodule = _set_submodule

DEVICE = "cuda"
torch.manual_seed(0)
ARM = globals().get("ARM", "it2it")   # "it2it" or "base2it"  <<< THE ONE SWITCH
MODEL_2B = "google/gemma-2-2b-it"   # RECIPIENT (26 layers, d_model 2304)
MODEL_9B = ("google/gemma-2-9b-it" if ARM == "it2it" else "google/gemma-2-9b")
# DONOR: 42 layers, d_model 3584, bf16 ~18 GB - fits with the 5 GB recipient on 48 GB.

# Smoke test lever, (True for 5 min test eval False for full eval)
SMOKE_TEST = False

# START GUESSES ONLY — scaled from the validated v1 depth ratio (~0.8 of recipient depth).
# CELL 7a/7b/7c re-derive the pair; if they suggest different layers, PASTE THEM HERE, re-run
# this cell and CELL 2b, then re-run 7a-7c to confirm CKA > ~0.8 before running anything else.
SINGLE_PAIR = (20, 34)   # validated v1 Gemma pair; CELL 7a/7b/7c re-derive it
LAYER_PAIRS = [(18, 34), (20, 37), (22, 41), (24, 44)]
L2_SINGLE, L9_SINGLE = SINGLE_PAIR
PATCH_POS = -1
RIDGE_LAMBDA = 1e3

if SMOKE_TEST:
    N_PATCH, N_ARITH_TRAIN, N_ARITH_EVAL = 12, 200, 120
    GSM_FIT, GSM_EVAL, GSM_STRENGTHS = 20, 24, [1.0]
    TASK_SEEDS, TASK_EPOCHS = [0, 1], 2
    BOOT_B = 1000
else:
    N_PATCH, N_ARITH_TRAIN, N_ARITH_EVAL = 150, 3000, 2000   # 3000/2000 matches v1 exactly
    GSM_FIT, GSM_EVAL, GSM_STRENGTHS = 150, 400, [1.0]       # GSM8K SUBSAMPLED to 400 (27B donor)
    TASK_SEEDS, TASK_EPOCHS = [0, 1, 2, 3, 4], 6             # 5 seeds for the task map
    BOOT_B = 10000

# BATCH SIZES: the only compute knob that differs between the two notebooks, because the
# resident weights differ. 27B donor + 2B recipient ~= 60 GiB of weights, so ARITH_BATCH is 16 and GSM_BATCH 8.
ARITH_BATCH, GSM_BATCH, MAX_NEW_GSM, MAX_NEW_ARITH = 16, 8, 300, 8
RESULTS = {}   # everything defensible gets concentrated here and printed at the end
print("SMOKE_TEST =", SMOKE_TEST, "| recipient:", MODEL_2B, "| donor:", MODEL_9B)
print("start-guess pair: recipient L%d <- donor L%d (RE-DERIVE in CELL 7)" % (L2_SINGLE, L9_SINGLE))


# IT prompting: default to the SAME plain few-shot prompt as v1 so every number stays
# directly comparable to the base-model results. The chat-template variant is available
# but OFF by default - it changes the prompt distribution and breaks comparability.
USE_CHAT_TEMPLATE = globals().get("USE_CHAT_TEMPLATE", False)
# NOT IMPLEMENTED: no prompt-building site in this notebook reads this flag (_aprompt in CELL 5
# and gsm_prompt in CELL 5 both build the plain v1 few-shot string unconditionally). Setting it
# True used to run the plain prompts anyway while printing "chat template: True", i.e. silently
# mislabelled results. Fail fast until the templating is actually wired into _aprompt/gsm_prompt.
assert not USE_CHAT_TEMPLATE, (
    "USE_CHAT_TEMPLATE=True is not implemented: _aprompt()/gsm_prompt() in CELL 5 do not apply "
    "tokenizer.apply_chat_template. Wire it in there first, or leave this False (the default, "
    "which keeps the prompt byte-identical to v1).")
print("ARM =", ARM, "| donor:", MODEL_9B, "| recipient:", MODEL_2B,
      "| chat template:", USE_CHAT_TEMPLATE)

SMOKE_TEST = False | recipient: google/gemma-2-2b-it | donor: google/gemma-2-9b-it
start-guess pair: recipient L20 <- donor L34 (RE-DERIVE in CELL 7)
ARM = it2it | donor: google/gemma-2-9b-it | recipient: google/gemma-2-2b-it | chat template: False


In [12]:
# === CELL 2b: experiment flags + donor-held-fixed settings (run right after CELL 2) ===
# Everything in this cell EXCEPT SAE_RELEASE and OUT_JSON is IDENTICAL in the 9B-recipient and
# 2B-recipient notebooks. That is deliberate: with the donor fixed, the two runs must differ
# only in the recipient, its batch size, and its layer pair.

# ---- experiment flags (each experiment is behind exactly one of these) ----
RUN_LAYERDERIV = True    # 1. layer derivation: patching + CKA + donor answer-probe sweep
RUN_CORE       = True    # 2. core channel: recon/task maps, shuffle + self-graft, first & full
RUN_PRESENCE   = False    # 2b. donor-presence probe (arith PRESENT vs GSM8K ABSENT)
RUN_CEIL       = True    # 3. free-vector ceiling
RUN_MLP        = True    # 4a. MLP (nonlinear) map — capacity from above
RUN_LOWRANK    = True    # 4b. low-rank truncation sweep — capacity from below
RUN_TRANSCRIBE = True    # 5. transcription probe (donor graft state + stitched vector)
RUN_INLP       = True    # 6. INLP answer-subspace erasure + matched-rank random control
RUN_RDELTA     = True    # 7. recipient-side SAE feature delta
RUN_DIGITS     = True    # 8. digit-1/2/3 probes at the donor graft site
RUN_DOSE       = False    # 9. depth dose-response
RUN_GSM        = False    # 10. GSM8K single-site stitch on the ~400-problem subsample

# ---- donor-held-fixed machinery (DO NOT change in only one of the two notebooks) ----
DONOR_BATCH = 8          # donor forwards use THIS, never ARITH_BATCH, so left-padding — and hence
                         # the bf16 numerics — are identical across the two notebooks and the
                         # donor-solved problem list comes out bit-for-bit the same.
L_DONOR_REF = 37         # fixed reference donor layer, probed in addition to the derived one
DONOR_SWEEP_LAYERS = [4, 8, 12, 16, 20, 24, 28, 32, 34, 36, 37, 38, 40, 42, 44, 45]
LAYERDERIV_PROBE_N = 600 # problems for the donor answer-probe sweep

# ---- per-experiment knobs (identical in both notebooks) ----
CEIL_N, CEIL_STEPS, CEIL_LR = 200, 150, 5e-2       # free-vector ceiling (subsampled: fwd+bwd heavy)
MLP_HIDDEN = 1024                                   # nonlinear map width
LOWRANK_RANKS = [1, 2, 4, 8, 16, 32, 64, 128]       # first-token sweep
LOWRANK_FULL_RANKS = [8, 32, 128]                   # full-answer at these ranks only (generation)
INLP_ROUNDS, INLP_CHECK_EVERY, COLLAPSE_FRAC = 60, 4, 0.5
ABLATE_FULL = True                                  # also score free-gen full answer after erasure
TOPK_DELTA = 30                                     # SAE: how many added/removed features to score
DOSE_STEPS, N_DOSE = [1, 2, 3, 4], 800              # depth dose-response
GSM_FIT_TOKENS = 12000                              # cap on token positions used to fit the GSM map
RUN_GSM_DONOR, GSM_DONOR_N = False, 200              # donor GSM8K solve rate (costly; donor is the
                                                    # same model in both notebooks, so it may be run
                                                    # once and copied — kept here so each JSON is
                                                    # self-contained)

# ---- the only genuinely recipient-specific settings ----
# GemmaScope coverage for INSTRUCTION-TUNED variants is not guaranteed. Try an IT release
# first; CELL 17 falls back to the base-model SAE with a loud caveat if it is unavailable.
SAE_RELEASE_IT   = "gemma-scope-2b-it-res-canonical"
SAE_RELEASE_BASE = "gemma-scope-2b-pt-res-canonical"
SAE_RELEASE = SAE_RELEASE_IT
SAE_ID_TEMPLATE = "layer_{layer}/width_16k/canonical"
OUT_JSON = f"it_results_{ARM}.json"

print("flags set | donor-side batch:", DONOR_BATCH, "| SAE:", SAE_RELEASE, "| out:", OUT_JSON)


# --- headline manifold test (CELL 13b) ---
RUN_DISTILL = True                       # KL + MSE-anchor sweep: the recon-vs-task axis
LAMBDAS     = [0.0, 0.1, 1.0, 10.0]      # 0.0 = pure KL (off-manifold); large = recon-like
KL_EPOCHS   = TASK_EPOCHS
KL_TEMP     = 1.0

flags set | donor-side batch: 8 | SAE: gemma-scope-2b-it-res-canonical | out: it_results_it2it.json


In [ ]:
# === CELL 3: Hugging Face login (Gemma is gated) ===
from huggingface_hub import login
login("[INSERT TOKEN HERE]")


In [14]:
# === CELL 4: statistics helpers (match the interval to the source of randomness) ===
def wilson(k, n, z=1.96):
    if n == 0: return (float("nan"),)*3
    p = k/n; d = 1 + z*z/n
    c = (p + z*z/(2*n))/d
    h = (z*math.sqrt(p*(1-p)/n + z*z/(4*n*n)))/d
    return p, max(0.0, c-h), min(1.0, c+h)
def wilson_bools(b, z=1.96):
    b = np.asarray(b, bool); return wilson(int(b.sum()), int(b.size), z)
def bootstrap_ci(x, B=None, alpha=0.05, seed=0):
    x = np.asarray(x, float); n = len(x); B = B or BOOT_B
    if n == 0: return (float("nan"),)*3
    rng = np.random.default_rng(seed)
    m = x[rng.integers(0, n, (B, n))].mean(1)
    lo, hi = np.percentile(m, [100*alpha/2, 100*(1-alpha/2)])
    return float(x.mean()), float(lo), float(hi)
_TC = {2:12.706,3:4.303,4:3.182,5:2.776,6:2.571,7:2.447,8:2.365,9:2.306,10:2.262}
def across_seed_ci(v, alpha=0.05):
    v = np.asarray(v, float); k = len(v); m = float(v.mean())
    if k < 2: return (m, float("nan"), float("nan"))
    se = v.std(ddof=1)/math.sqrt(k); t = _TC.get(k, 1.96)
    return (m, m-t*se, m+t*se)
def fmt(tr): p, lo, hi = tr; return f"{p:.3f} [{lo:.3f}, {hi:.3f}]"


In [15]:
# === CELL 5: shared helpers (hooks, padding, ridge map, problems, GSM8K, full-answer) ===
def _hid(o): return o[0] if isinstance(o, tuple) else o
def _pack(o, h): return (h,)+tuple(o[1:]) if isinstance(o, tuple) else h
def capture(store, key):
    def hook(_m,_i,o): store[key] = _hid(o)[:, PATCH_POS, :].detach()
    return hook
def patch_vec(vec):  # replace last-pos with vec (graph-safe: works under autograd too)
    def hook(_m,_i,o):
        h = _hid(o)
        if h.shape[1] == 0: return o
        h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
        return _pack(o, h2)
    return hook

def left_pad(id_list, pad_id):
    L = max(t.numel() for t in id_list)
    ids = torch.full((len(id_list), L), pad_id, dtype=torch.long)
    m = torch.zeros((len(id_list), L), dtype=torch.long)
    for i, t in enumerate(id_list):
        t = t.flatten(); ids[i, L-t.numel():] = t; m[i, L-t.numel():] = 1
    return ids, m

def fit_ridge(X9, X2, lam=RIDGE_LAMBDA):
    mu9, mu2 = X9.mean(0), X2.mean(0)
    A, B = X9-mu9, X2-mu2
    W = torch.linalg.solve(A.T@A + lam*torch.eye(A.shape[1]), A.T@B)
    return mu9, mu2, W
def apply_map(x, m): mu9, mu2, W = m; return (x-mu9)@W + mu2

# ---- arithmetic problems (muladd only: healthy unsolvable bin) ----
FEWSHOT = ("2 + 5 = 7\n6 * 3 = 18\n4 * 7 + 2 = 30\n9 * 8 = 72\n"
           "3 * 4 + 5 = 17\n40 * 20 = 800\n")
def _aprompt(expr): return f"{FEWSHOT}{expr} ="
def _aencode(tok, expr, ans):
    # Build prompt and prompt+answer, then target the FIRST answer token that carries a
    # digit. On Gemma the answer tokenizes as [space, digit] so that token is at len(p)+1;
    # on byte-level BPE tokenizers (Qwen, Llama) the leading space fuses with the first
    # digit, so it's at len(p). Scanning for the first digit-bearing token handles both,
    # plus any tokenizer that emits leading whitespace/markup tokens before the number.
    p = tok(_aprompt(expr)).input_ids
    f = tok(_aprompt(expr) + " " + str(ans)).input_ids
    if f[:len(p)] != p or len(f) <= len(p): return None, None
    j = len(p)
    while j < len(f) and not any(ch.isdigit() for ch in tok.decode([f[j]])): j += 1
    if j >= len(f): return None, None
    return torch.tensor(f[:j]), f[j]   # context up to (not incl.) the first digit token; target = that token
def gen_arith(tok, n, rng, exclude=None):
    exclude = exclude or set(); out, seen = [], set()
    tries = 0
    while len(out) < n and tries < n*120:
        tries += 1
        a, b, c = rng.randint(2, 40), rng.randint(2, 40), rng.randint(1, 99)
        expr, ans = f"{a} * {b} + {c}", a*b+c
        if expr in seen or expr in exclude: continue
        seen.add(expr)
        ids, tok_id = _aencode(tok, expr, ans)
        if tok_id is None: continue
        out.append(dict(expr=expr, ans=ans, ids=ids, tok=tok_id))
    return out

# ---- GSM8K: validated prompt + extraction (from the Linear_gsm8k notebook) ----
import re as _re
def gsm_prompt(q):
    return (
        "Below are math problems with detailed step-by-step solutions.\n\n"
        "Problem: Natalia sold clips to 48 of her friends in April, and then she sold "
        "half as many clips in May. How many clips did Natalia sell altogether in April and May?\n"
        "Solution: Let's think step-by-step.\n"
        "1. Clips sold in April: 48\n"
        "2. Clips sold in May: 48 / 2 = 24\n"
        "3. Total clips: 48 + 24 = 72\n"
        "#### 72\n\n"
        f"Problem: {q}\n"
        "Solution: Let's think step-by-step."
    )
def gsm_extract(text):
    m = _re.search(r"####\s*(-?[\d,.]+)", text)
    if m:
        try: return float(m.group(1).replace(",", ""))
        except ValueError: pass
    m = _re.search(r"answer is\s*(-?[\d,.]+)", text, _re.IGNORECASE)
    if m:
        try: return float(m.group(1).replace(",", ""))
        except ValueError: pass
    nums = _re.findall(r"-?[\d,.]+", text)
    if nums:
        try: return float(nums[-1].rstrip(".").replace(",", ""))
        except ValueError: return None
    return None
def _parse_first_int(text):
    """Arithmetic answers: take the FIRST integer the model emits after '=',
    not the last (the model may continue with few-shot-style lines)."""
    m = _re.search(r"-?\d+", text)
    return float(m.group()) if m else None

# ---- generic batched forward: last-pos resid at given layers + top token ----
@torch.inference_mode()
def states_and_top(model, layers, prob_ids, batch=ARITH_BATCH):
    base = model.model if hasattr(model, "model") else model
    acc = {L: [] for L in layers}; top = []
    for i in range(0, len(prob_ids), batch):
        ids, m = left_pad(prob_ids[i:i+batch], tokenizer.pad_token_id)
        out = model(ids.to(DEVICE), attention_mask=m.to(DEVICE), output_hidden_states=True)
        for L in layers: acc[L].append(out.hidden_states[L+1][:, -1, :].float().cpu())
        top += out.logits[:, -1, :].argmax(-1).cpu().tolist()
    return {L: torch.cat(v) for L, v in acc.items()}, top

# ---- per-batch graft hook: replace last prompt-position resid with vec[B,d] ----
# Fires only during prefill (seq len > 1); no-ops during generation (len==1) and
# when the batch dim doesn't match, so generation proceeds normally after seeding.
_graft = {"vec": None}
def patch_vec_batch(_m, _i, o):
    h = _hid(o); vec = _graft["vec"]
    if vec is None or h.shape[1] <= 1 or h.shape[0] != vec.shape[0]:
        return o
    h2 = torch.cat([h[:, :-1, :], vec.to(h.dtype).unsqueeze(1)], 1)
    return _pack(o, h2)

@torch.inference_mode()
def arith_fullanswer_correct(model, layer, probs, vecs=None, batch=ARITH_BATCH):
    """Generate the full number and compare to gold. If vecs is given, vecs[i] is
    grafted at the last prompt position of problem i (prefill) before generation.
    Returns list[bool], one per problem."""
    ok, handle = [], None
    if vecs is not None:
        handle = model.model.layers[layer].register_forward_hook(patch_vec_batch)
    try:
        for i in range(0, len(probs), batch):
            chunk = probs[i:i+batch]
            ids, m = left_pad([p["ids"] for p in chunk], tokenizer.pad_token_id)
            ids, m = ids.to(DEVICE), m.to(DEVICE)
            _graft["vec"] = (torch.stack(vecs[i:i+len(chunk)]).to(DEVICE)
                             if vecs is not None else None)
            gen = model.generate(ids, attention_mask=m, max_new_tokens=MAX_NEW_ARITH,
                                 do_sample=False, pad_token_id=tokenizer.eos_token_id)
            txt = tokenizer.batch_decode(gen[:, ids.shape[1]:], skip_special_tokens=True)
            for p, t in zip(chunk, txt):
                pred = _parse_first_int(t)
                ok.append(pred is not None and abs(pred - p["ans"]) < 0.5)
    finally:
        if handle: handle.remove()
        _graft["vec"] = None
    return ok

print("helpers defined")


helpers defined


In [16]:
# === CELL 5b: extra helpers used by the experiment cells (probes, digits, datasets) ===
# fd() is defined HERE on purpose: the base notebooks only define it inside a late experiment
# cell, and several cells below call it earlier. Everything an experiment cell needs that is
# not in CELL 4 / CELL 5 lives in this cell, so there is exactly one place to look.
import collections
from datasets import load_dataset

def fd(x):
    """Leading (first) digit of an integer answer, 0-9."""
    return int(str(abs(int(round(x))))[0]) if x is not None else 0

def _dig(ans, k):
    """k-th digit (1-indexed from the left) of an integer answer, or None if too short."""
    s = str(abs(int(ans)))
    return int(s[k-1]) if len(s) >= k else None

def fit_probe_w(X, y, nclass=10, steps=300, lr=1e-2):
    """Multinomial linear probe by full-batch Adam. Returns (weights, feature mean).
    Runs on whatever device X is on, so it serves both the CPU state probes and the
    on-DEVICE INLP loop."""
    dev = X.device
    Pw = torch.zeros(X.shape[1], nclass, device=dev, requires_grad=True)
    opt = torch.optim.Adam([Pw], lr=lr)
    mu = X.mean(0).detach(); Xc = (X - mu).detach(); yy = y.to(dev)
    for _ in range(steps):
        opt.zero_grad(); F.cross_entropy(Xc @ Pw, yy).backward(); opt.step()
    return Pw.detach(), mu

def probe_first_digit(Xtr, ytr, Xte, yte=None, steps=300):
    """Fit on (Xtr,ytr), return PREDICTIONS on Xte (cpu long tensor). yte is accepted and
    ignored so the call signature matches the base notebooks verbatim."""
    Pw, mu = fit_probe_w(Xtr, ytr, steps=steps)
    return ((Xte.to(Pw.device) - mu) @ Pw).argmax(1).cpu()

def probe_split_bools(X, y, steps=300):
    """Half/half split of one state matrix -> list[bool] of test-half correctness."""
    h = len(y) // 2
    pred = probe_first_digit(X[:h], y[:h], X[h:], steps=steps)
    return (pred == y[h:].cpu()).tolist()

def majority_acc(y):
    y = torch.as_tensor(y)
    if y.numel() == 0: return float("nan")
    maj = collections.Counter(y.tolist()).most_common(1)[0][0]
    return round(float((y == maj).float().mean()), 3)

print("extra helpers defined (fd, _dig, fit_probe_w, probe_first_digit, probe_split_bools, majority_acc)")


extra helpers defined (fd, _dig, fit_probe_w, probe_first_digit, probe_split_bools, majority_acc)


In [17]:
# === CELL 6: load both models once; donor in bf16 by default (4-bit optional) ===
tokenizer = AutoTokenizer.from_pretrained(MODEL_2B)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

# QUANTIZE_9B: set True only when the donor won't fit in bf16 (e.g. the original Gemma-2-9B on
# a small card). For 7-8B donors (Qwen-7B ~15GB, Llama-8B ~16GB) on a 20GB+ card, keep this
# False — bf16 is correct and avoids a serious failure mode: under 4-bit, this transformers/bnb
# build runs unquantized layers in FP16, and Qwen/Llama activations OVERFLOW fp16 (>65504) ->
# NaN logits -> argmax collapses to token 0 ('!'). bf16 has the exponent range to avoid this.
# *** IN THIS NOTEBOOK THE DONOR IS gemma-2-27b AND MUST STAY bf16. *** Donor state fidelity at
# the graft site is the object of study, and 4-bit on this build reproduces exactly the fp16
# overflow -> NaN failure above. Do not flip this to True to "save VRAM"; get an 80 GB card.
QUANTIZE_9B = globals().get("QUANTIZE_9B", False)
assert QUANTIZE_9B is False, "the 27B donor must be loaded in bf16 — see the comment above"

model_2b = AutoModelForCausalLM.from_pretrained(
    MODEL_2B, torch_dtype=torch.bfloat16, attn_implementation="eager",
    low_cpu_mem_usage=True).to(DEVICE).eval()
if QUANTIZE_9B:
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_use_double_quant=True)
    model_9b = AutoModelForCausalLM.from_pretrained(
        MODEL_9B, quantization_config=bnb, device_map={"": 0},
        torch_dtype=torch.bfloat16, attn_implementation="eager", low_cpu_mem_usage=True).eval()
else:
    model_9b = AutoModelForCausalLM.from_pretrained(
        MODEL_9B, torch_dtype=torch.bfloat16, attn_implementation="eager",
        low_cpu_mem_usage=True).to(DEVICE).eval()
print("loaded:", model_2b.config.num_hidden_layers, "x recipient layers,",
      model_9b.config.num_hidden_layers, "x donor layers |",
      "donor quantized" if QUANTIZE_9B else "donor bf16")

# Health check: catch a NaN/overflow blowup (the fp16-under-4bit failure) immediately, not
# 168 silently-filtered pairs later. A healthy donor tops a real word here, never token 0.
with torch.inference_mode():
    _hl = model_9b(tokenizer("The capital of France is", return_tensors="pt").to(DEVICE).input_ids).logits[0, -1, :]
assert not torch.isnan(_hl).any() and not torch.isinf(_hl).any(), (
    "donor produced NaN/Inf logits — numerical blowup. If QUANTIZE_9B=True, the donor is overflowing "
    "fp16; set QUANTIZE_9B=False to load it in bf16 (needs the VRAM but is numerically safe).")
del _hl

# Same-family requirement: the two models must share the TOKENIZER so positions align
# (the whole stitch grafts by position). NOTE: config.vocab_size is the *padded embedding*
# count, not the tokenizer — Qwen pads differently across sizes (0.5B=151936, 7B=152064)
# while sharing one tokenizer, so comparing config.vocab_size gives false alarms. Verify the
# tokenizer itself instead, by checking a probe string maps to identical ids under each model's
# own tokenizer. (We load one shared tokenizer, but this also catches an accidental mismatch.)
_tk2 = AutoTokenizer.from_pretrained(MODEL_2B); _tk9 = AutoTokenizer.from_pretrained(MODEL_9B)
_probe = "3 * 12 + 7 = 43\nThe answer is 256."
assert _tk2(_probe).input_ids == _tk9(_probe).input_ids, (
    "tokenizer mismatch: the two models tokenize the same text differently, so positions won't "
    "align. This notebook requires a SAME-FAMILY pair sharing one tokenizer.")
del _tk2, _tk9

D_RECIP = model_2b.config.hidden_size
D_DONOR = model_9b.config.hidden_size
NL_RECIP = model_2b.config.num_hidden_layers
NL_DONOR = model_9b.config.num_hidden_layers
print(f"d_model: recipient {D_RECIP}, donor {D_DONOR} | layers: recipient {NL_RECIP}, donor {NL_DONOR}")
print(f"VRAM in use: {torch.cuda.memory_allocated()/2**30:.1f} GiB allocated, "
      f"{torch.cuda.mem_get_info()[0]/2**30:.1f} GiB free")


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

d_model: recipient 2304, donor 3584 | layers: recipient 26, donor 42
VRAM in use: 22.1 GiB allocated, 21.7 GiB free


In [18]:
# === CELL 7 (EXP1a): activation patching on the RECIPIENT + CKA against DONOR layers ===
# Base: CELL 7 of consolidated_eval.ipynb. This is HALF of the layer-derivation experiment;
# CELL 7b (donor answer-probe sweep) is the other half. Re-derive the pair here — the
# SINGLE_PAIR/LAYER_PAIRS values in CELL 2 are only depth-ratio START GUESSES.
RUN_LAYERDERIV = globals().get("RUN_LAYERDERIV", True)
if RUN_LAYERDERIV:
    def _add_prompt(a, b): return f"2 + 5 = 7\n8 + 1 = 9\n4 + 3 = 7\n{a} + {b} ="
    def _enc_full(a, b):
        p = tokenizer(_add_prompt(a, b)).input_ids
        f = tokenizer(_add_prompt(a, b) + " " + str(a + b)).input_ids
        if f[:len(p)] != p or len(f) <= len(p): return None, None
        return p, f
    def build_add_pairs(n):
        # Use sums whose answers are SINGLE digits (2..9) so the first answer token IS the whole
        # answer: clean a+b and corrupted a+c then differ at the very first answer token, which is
        # exactly where the arithmetic is computed.
        combos = [(a,b,c) for a in range(1,9) for b in range(1,9) for c in range(1,9)
                  if c != b and 2 <= a+b <= 9 and 2 <= a+c <= 9 and (a+b) != (a+c)]
        random.Random(0).shuffle(combos); pairs = []
        for a, b, c in combos:
            pb, fb = _enc_full(a, b); pc, fc = _enc_full(a, c)
            if pb is None or pc is None or len(pb) != len(pc): continue
            Lp = len(pb)
            ab, ac = fb[Lp:], fc[Lp:]                  # the answer tokens of clean vs corrupted
            k = 0                                       # skip any shared leading answer tokens (e.g. a space)
            while k < len(ab) and k < len(ac) and ab[k] == ac[k]: k += 1
            if k >= len(ab) or k >= len(ac) or ab[k] == ac[k]: continue
            ci, ri = torch.tensor([fb[:Lp+k]]), torch.tensor([fc[:Lp+k]])
            if ci.shape[1] != ri.shape[1]: continue     # context up to the first diverging answer token
            pairs.append(dict(clean=ci, corr=ri, ct=ab[k], rt=ac[k]))
            if len(pairs) >= n: break
        return pairs

    def _ld(lg, ct, rt): return (lg[ct]-lg[rt]).item()
    @torch.inference_mode()
    def patch_recovery(model, pairs, tag=""):
        nl = model.config.num_hidden_layers; rows = []; cka = {L: [] for L in range(nl)}
        for pi, pr in enumerate(pairs):
            if pi % 25 == 0: print(f"    [{tag}] patch pair {pi}/{len(pairs)}", flush=True)
            ci, ri = pr["clean"].to(DEVICE), pr["corr"].to(DEVICE); ct, rt = pr["ct"], pr["rt"]
            cache = {}; hs = [model.model.layers[L].register_forward_hook(capture(cache, L)) for L in range(nl)]
            try: clg = model(ci).logits[0, -1, :]
            finally:
                for h in hs: h.remove()
            for L in range(nl): cka[L].append(cache[L][0].float().cpu())
            rlg = model(ri).logits[0, -1, :]
            cld, rld = _ld(clg, ct, rt), _ld(rlg, ct, rt)
            if clg.argmax().item()!=ct or rlg.argmax().item()!=rt or abs(cld-rld)<1e-6: continue
            rec = np.zeros(nl)
            for L in range(nl):
                h = model.model.layers[L].register_forward_hook(patch_vec(cache[L]))
                try: plg = model(ri).logits[0, -1, :]
                finally: h.remove()
                rec[L] = (_ld(plg, ct, rt) - rld) / (cld - rld)
            rows.append(rec)
        return np.array(rows), {L: torch.stack(v) for L, v in cka.items()}
    def linear_cka(X, Y):
        X, Y = X-X.mean(0, keepdim=True), Y-Y.mean(0, keepdim=True)
        hsic = (X.t()@Y).pow(2).sum()
        return (hsic / (torch.sqrt((X.t()@X).pow(2).sum())*torch.sqrt((Y.t()@Y).pow(2).sum()))).item()

    pairs = build_add_pairs(N_PATCH)
    assert len(pairs) >= 4, (f"only built {len(pairs)} addition pairs — this tokenizer likely splits "
                             "numbers unusually; EXP1 needs a handful of valid clean/corrupted pairs.")
    rec2, cka2 = patch_recovery(model_2b, pairs, tag="recipient")
    rec9, cka9 = patch_recovery(model_9b, pairs, tag="donor-27B")
    print(f"patch pairs built={len(pairs)} | survived filter: recipient={len(rec2)}, donor={len(rec9)}")
    assert len(rec2) > 0 and len(rec9) > 0, (
        f"no pairs survived for {'recipient' if len(rec2)==0 else 'donor'} "
        f"(recipient={len(rec2)}, donor={len(rec9)}): that model isn't greedily solving the "
        "single-digit sums at the first token.")
    # exclude the readout layers (trivially recovery 1.0) when reporting the causal peak
    peak2 = int(rec2.mean(0)[:-2].argmax()); peak9 = int(rec9.mean(0)[:-2].argmax())
    # validate the CONFIGURED graft layer: CKA(recipient L2_SINGLE, donor layer L) over donor layers
    ckas = [linear_cka(cka2[L2_SINGLE], cka9[L]) for L in range(NL_DONOR)]
    best9 = int(np.argmax(ckas))
    RESULTS["patching"] = {
        "n_used_recipient": len(rec2), "n_used_donor": len(rec9),
        "causal_peak_recipient": peak2, "causal_peak_donor": peak9,
        "graft_layer_recipient": L2_SINGLE, "recovery_recipient_at_graft": fmt(bootstrap_ci(rec2[:, L2_SINGLE])),
        "best_donor_match_for_graft": best9, "cka_at_graft": round(ckas[best9], 3),
        "cka_at_configured_pair": round(linear_cka(cka2[L2_SINGLE], cka9[L9_SINGLE]), 3),
        "recovery_curve_recipient": [round(x, 2) for x in rec2.mean(0).tolist()],
        "recovery_curve_donor": [round(x, 2) for x in rec9.mean(0).tolist()]}
    print("EXP1a:", json.dumps({k: v for k, v in RESULTS["patching"].items() if "curve" not in k}, indent=2))
    print("  recipient recovery by layer:", RESULTS["patching"]["recovery_curve_recipient"])
    print("  donor     recovery by layer:", RESULTS["patching"]["recovery_curve_donor"])
    # --- auto-suggest layers for THIS pair (paste into CELL 2 when running a NEW pair) ---
    def _best9(a): return int(np.argmax([linear_cka(cka2[a], cka9[L]) for L in range(NL_DONOR)]))
    r2 = rec2.mean(0)[:-1]                                  # drop the trivial readout layer
    band2 = np.where(r2 > 0.5 * r2.max())[0]                # the recipient's causal band
    graft2 = int(band2[len(band2) // 2])                    # middle of the band (safe graft layer)
    sug_band2 = sorted({int(x) for x in np.linspace(band2[0], band2[-1], 4).round()})
    SUGGESTED_SINGLE_PAIR = (graft2, _best9(graft2))
    SUGGESTED_LAYER_PAIRS = [(a, _best9(a)) for a in sug_band2]
    RESULTS["patching"]["suggested_single_pair"] = list(SUGGESTED_SINGLE_PAIR)
    RESULTS["patching"]["suggested_layer_pairs"] = [list(p) for p in SUGGESTED_LAYER_PAIRS]
    print("\nSUGGESTED LAYERS FOR THIS PAIR -> paste into CELL 2, then re-run CELL 2 and this cell:")
    print(f"  SINGLE_PAIR = {SUGGESTED_SINGLE_PAIR}")
    print(f"  LAYER_PAIRS = {SUGGESTED_LAYER_PAIRS}")
    print(f"  (sanity: CKA at suggested pair = "
          f"{round(linear_cka(cka2[graft2], cka9[_best9(graft2)]), 3)}; want > ~0.8)")
else:
    print("EXP1a skipped (RUN_LAYERDERIV=False).")


    [recipient] patch pair 0/150
    [recipient] patch pair 25/150
    [recipient] patch pair 50/150
    [recipient] patch pair 75/150
    [recipient] patch pair 100/150
    [recipient] patch pair 125/150
    [donor-27B] patch pair 0/150
    [donor-27B] patch pair 25/150
    [donor-27B] patch pair 50/150
    [donor-27B] patch pair 75/150
    [donor-27B] patch pair 100/150
    [donor-27B] patch pair 125/150
patch pairs built=150 | survived filter: recipient=150, donor=150
EXP1a: {
  "n_used_recipient": 150,
  "n_used_donor": 150,
  "causal_peak_recipient": 23,
  "causal_peak_donor": 39,
  "graft_layer_recipient": 20,
  "recovery_recipient_at_graft": "0.748 [0.738, 0.757]",
  "best_donor_match_for_graft": 34,
  "cka_at_graft": 0.944,
  "cka_at_configured_pair": 0.944
}
  recipient recovery by layer: [-0.0, 0.0, -0.0, -0.0, -0.0, 0.0, -0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.01, 0.01, 0.07, 0.23, 0.25, 0.72, 0.72, 0.75, 0.76, 0.76, 0.78, 0.8, 1.0]
  donor     recovery by layer: [-0.0, 0.0, 0

In [19]:
# === CELL 7b (EXP1b): DONOR answer-probe sweep — where does the 27B hold the answer? ===
# Base: CELL 15 of consolidated_eval.ipynb + the probe half of CELL G2 of Followup2_Gemma.ipynb.
# Self-contained (its own problem set) so it can run BEFORE the main arithmetic setup and feed
# the layer choice. Sweep layers are FIXED and IDENTICAL in the 9B- and 2B-recipient notebooks,
# so the two donor curves are directly joinable — the donor is the same model, so they should
# agree to within probe noise. That agreement is itself a check that the runs are comparable.
if RUN_LAYERDERIV:
    dsw = gen_arith(tokenizer, LAYERDERIV_PROBE_N, random.Random(7))
    print(f"donor probe sweep on {len(dsw)} arithmetic problems, layers {DONOR_SWEEP_LAYERS}")
    sweep_layers = sorted(set(DONOR_SWEEP_LAYERS) | {L9_SINGLE})
    sweep_layers = [L for L in sweep_layers if 0 <= L < NL_DONOR]
    X9_sweep, dsw_top = states_and_top(model_9b, sweep_layers, [p["ids"] for p in dsw], batch=DONOR_BATCH)
    y_sw = torch.tensor([fd(p["ans"]) for p in dsw])
    donor_first_acc_sweepset = float(np.mean([dsw_top[i] == dsw[i]["tok"] for i in range(len(dsw))]))
    sweep = {}
    for L in sweep_layers:
        sweep[f"L{L}"] = fmt(wilson_bools(probe_split_bools(X9_sweep[L], y_sw)))
    RESULTS["donor_layer_probe_sweep"] = {
        "n": len(dsw), "layers": sweep_layers, "graft_layer_donor": L9_SINGLE,
        "donor_first_token_acc_on_sweepset": round(donor_first_acc_sweepset, 3),
        "majority_baseline": majority_acc(y_sw[len(y_sw)//2:]),
        "digit1_probe_by_layer": sweep}
    print("EXP1b donor layer probe sweep:", json.dumps(RESULTS["donor_layer_probe_sweep"], indent=2))
    del X9_sweep
else:
    print("EXP1b skipped (RUN_LAYERDERIV=False).")


donor probe sweep on 600 arithmetic problems, layers [4, 8, 12, 16, 20, 24, 28, 32, 34, 36, 37, 38, 40, 42, 44, 45]
EXP1b donor layer probe sweep: {
  "n": 600,
  "layers": [
    4,
    8,
    12,
    16,
    20,
    24,
    28,
    32,
    34,
    36,
    37,
    38,
    40
  ],
  "graft_layer_donor": 34,
  "donor_first_token_acc_on_sweepset": 0.87,
  "majority_baseline": 0.287,
  "digit1_probe_by_layer": {
    "L4": "0.223 [0.180, 0.274]",
    "L8": "0.263 [0.217, 0.316]",
    "L12": "0.280 [0.232, 0.333]",
    "L16": "0.290 [0.242, 0.344]",
    "L20": "0.347 [0.295, 0.402]",
    "L24": "0.340 [0.289, 0.395]",
    "L28": "0.370 [0.317, 0.426]",
    "L32": "0.650 [0.594, 0.702]",
    "L34": "0.743 [0.691, 0.789]",
    "L36": "0.797 [0.748, 0.838]",
    "L37": "0.823 [0.776, 0.862]",
    "L38": "0.843 [0.798, 0.880]",
    "L40": "0.870 [0.827, 0.903]"
  }
}


In [20]:
# === CELL 7c (EXP1c): CKA table for the candidate pairs (run after 7a; cheap) ===
# Inspect neighbours of the configured pair before committing. If a neighbour beats the
# configured pair on CKA *and* sits inside the recipient's causal band, prefer it: edit
# SINGLE_PAIR/LAYER_PAIRS in CELL 2, re-run CELL 2, then re-run 7a/7b/7c.
if RUN_LAYERDERIV and "cka2" in globals():
    cand = sorted({(L2_SINGLE, L9_SINGLE)} | set(LAYER_PAIRS) | set(SUGGESTED_LAYER_PAIRS)
                  | {tuple(SUGGESTED_SINGLE_PAIR)}
                  | {(L2_SINGLE + da, L9_SINGLE + dd) for da in (-2, -1, 0, 1, 2) for dd in (-3, -1, 0, 1, 3)})
    cand = [(a, b) for (a, b) in cand if 0 <= a < NL_RECIP and 0 <= b < NL_DONOR]
    tab = {}
    for a, b in cand:
        tab[f"recipL{a}<-donorL{b}"] = {
            "cka": round(linear_cka(cka2[a], cka9[b]), 3),
            "recip_recovery": round(float(rec2.mean(0)[a]), 3),
            "donor_recovery": round(float(rec9.mean(0)[b]), 3)}
    for k in sorted(tab, key=lambda k: -tab[k]["cka"])[:20]:
        print(f"  {k:26s} CKA={tab[k]['cka']:.3f}  recip_rec={tab[k]['recip_recovery']:.3f}  "
              f"donor_rec={tab[k]['donor_recovery']:.3f}")
    RESULTS["cka_candidate_table"] = tab
    _cfg_cka = tab[f"recipL{L2_SINGLE}<-donorL{L9_SINGLE}"]["cka"]
    print(f"\nCONFIGURED PAIR IN USE: recipient L{L2_SINGLE} <- donor L{L9_SINGLE} (CKA {_cfg_cka:.3f})")
    # free the per-layer patching caches before the heavy cells (they are large on a 46-layer donor)
    del cka2, cka9
    import gc; gc.collect(); torch.cuda.empty_cache()
else:
    print("EXP1c skipped (RUN_LAYERDERIV=False, or the CKA caches were already freed — "
          "re-run CELL 7a if you need this table again).")


  recipL24<-donorL39         CKA=0.972  recip_rec=0.804  donor_rec=0.596
  recipL21<-donorL35         CKA=0.952  recip_rec=0.760  donor_rec=0.546
  recipL22<-donorL35         CKA=0.951  recip_rec=0.761  donor_rec=0.546
  recipL21<-donorL34         CKA=0.948  recip_rec=0.760  donor_rec=0.543
  recipL20<-donorL34         CKA=0.944  recip_rec=0.748  donor_rec=0.543
  recipL22<-donorL37         CKA=0.939  recip_rec=0.761  donor_rec=0.547
  recipL22<-donorL34         CKA=0.936  recip_rec=0.761  donor_rec=0.543
  recipL20<-donorL33         CKA=0.930  recip_rec=0.748  donor_rec=0.543
  recipL20<-donorL35         CKA=0.930  recip_rec=0.748  donor_rec=0.546
  recipL21<-donorL37         CKA=0.921  recip_rec=0.760  donor_rec=0.547
  recipL19<-donorL31         CKA=0.920  recip_rec=0.722  donor_rec=0.536
  recipL19<-donorL33         CKA=0.915  recip_rec=0.722  donor_rec=0.543
  recipL21<-donorL33         CKA=0.902  recip_rec=0.760  donor_rec=0.543
  recipL18<-donorL32         CKA=0.900  recip_rec=0

In [21]:
# === CELL 8: EXP2 setup — arithmetic states, native correctness, reconstruction map ===
# Base: CELL 8 of consolidated_eval.ipynb, plus the DONOR-SIDE bookkeeping the cross-notebook
# comparison needs. Generator, seeds and n match v1 exactly (3000 train / 2000 eval, a*b+c).
#
# *** DONOR-HELD-FIXED INVARIANT ***
# `train`/`evalp` come from gen_arith with fixed seeds and the SHARED Gemma-2 tokenizer, and the
# donor-solved filter uses only the DONOR's own argmax at prefill. Donor forwards use DONOR_BATCH
# (NOT ARITH_BATCH), which is identical in both notebooks, so left-padding — and therefore the
# bf16 numerics — are identical too. Consequence: DONOR_SOLVED_EXPRS below is bit-for-bit the same
# list in the 9B-recipient and 2B-recipient notebooks. Any conferral difference between the two is
# therefore attributable to P(extract | present), not to a different donor-side sample.
train = gen_arith(tokenizer, N_ARITH_TRAIN, random.Random(0))
evalp = gen_arith(tokenizer, N_ARITH_EVAL, random.Random(1), {p["expr"] for p in train})
print(f"arith: {len(train)} train, {len(evalp)} eval")
N_EVAL_ALL = len(evalp)

X9t, _ = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in train], batch=DONOR_BATCH); X9t = X9t[L9_SINGLE]
X9e, nine_top = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in evalp], batch=DONOR_BATCH); X9e = X9e[L9_SINGLE]
keep = [i for i in range(len(evalp)) if nine_top[i] == evalp[i]["tok"]]
DONOR_SOLVE_RATE = len(keep) / max(N_EVAL_ALL, 1)
DONOR_SOLVED_EXPRS = [evalp[i]["expr"] for i in keep]
evalp = [evalp[i] for i in keep]; X9e = X9e[keep]
print(f"  donor solves {len(evalp)}/{N_EVAL_ALL} = {DONOR_SOLVE_RATE:.3f} (this is P(donor solves in one pass))")

X2t, _ = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in train]); X2t = X2t[L2_SINGLE]
X2e, two_top = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in evalp]); X2e = X2e[L2_SINGLE]
solvable = [two_top[i] == evalp[i]["tok"] for i in range(len(evalp))]
unsolv = [i for i in range(len(evalp)) if not solvable[i]]
solv = [i for i in range(len(evalp)) if solvable[i]]
print(f"  recipient natively solves {len(solv)}/{len(evalp)} (UNSOLVABLE BIN n={len(unsolv)})")

mu9, mu2, Wr = fit_ridge(X9t, X2t)
recon_map = (mu9.to(DEVICE), mu2.to(DEVICE), Wr.to(DEVICE))
def map_recon(x9): m9, m2, W = recon_map; return (x9.to(DEVICE)-m9)@W + m2

RESULTS["bins"] = {
    "n_train": len(train), "n_eval_generated": N_EVAL_ALL,
    "donor_solve_rate": round(DONOR_SOLVE_RATE, 4), "n_donor_solved": len(evalp),
    "n_recipient_solvable": len(solv), "n_unsolvable_bin": len(unsolv),
    "recipient_native_rate_on_donor_solved": round(len(solv)/max(len(evalp),1), 4)}
print("bins:", json.dumps(RESULTS["bins"], indent=2))


arith: 3000 train, 2000 eval
  donor solves 1805/2000 = 0.902 (this is P(donor solves in one pass))
  recipient natively solves 1445/1805 (UNSOLVABLE BIN n=360)
bins: {
  "n_train": 3000,
  "n_eval_generated": 2000,
  "donor_solve_rate": 0.9025,
  "n_donor_solved": 1805,
  "n_recipient_solvable": 1445,
  "n_unsolvable_bin": 360,
  "recipient_native_rate_on_donor_solved": 0.8006
}


In [22]:
# === CELL 9: EXP3 — train the task-supervised map, 5 seeds (the only stochastic part) ===
mu9d, mu2d = mu9.to(DEVICE), mu2.to(DEVICE)
task_maps = []
model_2b.requires_grad_(False)
for seed in TASK_SEEDS:
    torch.manual_seed(seed); random.seed(seed)
    W = Wr.clone().to(DEVICE).requires_grad_(True); b = mu2.clone().to(DEVICE).requires_grad_(True)
    opt = torch.optim.Adam([W, b], lr=1e-3)
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    idx = list(range(len(train)))
    try:
        for ep in range(TASK_EPOCHS):
            random.Random(seed*100+ep).shuffle(idx)
            for s in range(0, len(idx), ARITH_BATCH):
                sub = idx[s:s+ARITH_BATCH]
                x9 = X9t[sub].to(DEVICE)
                _graft["vec"] = (x9 - mu9d) @ W + b
                ids, m = left_pad([train[k]["ids"] for k in sub], tokenizer.pad_token_id)
                lg = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                tgt = torch.tensor([train[k]["tok"] for k in sub], device=DEVICE)
                loss = F.cross_entropy(lg, tgt); opt.zero_grad(); loss.backward(); opt.step()
    finally:
        handle.remove(); _graft["vec"] = None
    task_maps.append((W.detach(), b.detach()))
    print(f"  seed {seed}: final batch CE {loss.item():.3f}", flush=True)
model_2b.requires_grad_(True)


  seed 0: final batch CE 0.035
  seed 1: final batch CE 0.565
  seed 2: final batch CE 0.045
  seed 3: final batch CE 0.130
  seed 4: final batch CE 0.044


Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
          (rotary_emb): Gemma2RotaryEmbedding()
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_layernorm): Gemma2RMSNorm((2304,), eps

In [23]:
# === CELL 10: EXP2+3 eval — reconstruct vs task, first-token AND full-answer, with CIs ===
@torch.inference_mode()
def first_token_confer(map_fn, idxs):
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    out = []
    try:
        for i in range(0, len(idxs), ARITH_BATCH):
            sub = idxs[i:i+ARITH_BATCH]
            _graft["vec"] = map_fn(X9e[sub])
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == evalp[sub[k]]["tok"] for k in range(len(sub))]
    finally:
        handle.remove(); _graft["vec"] = None
    return out
def full_confer(map_fn, idxs):
    vecs = list(map_fn(X9e[idxs]).cpu())
    return arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in idxs], vecs=vecs)

def map_task(i):
    W, b = task_maps[i]
    return lambda x9: (x9.to(DEVICE)-mu9d)@W + b
shuf = torch.randperm(len(evalp))
@torch.inference_mode()
def shuffle_confer(idxs):  # task map fed the WRONG problem's donor state (specificity control)
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch); out = []
    W, b = task_maps[0]
    try:
        for i in range(0, len(idxs), ARITH_BATCH):
            sub = idxs[i:i+ARITH_BATCH]
            donor = X9e[shuf[i:i+len(sub)]].to(DEVICE)
            _graft["vec"] = (donor - mu9d)@W + b
            ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
            top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
            out += [top[k] == evalp[sub[k]]["tok"] for k in range(len(sub))]
    finally:
        handle.remove(); _graft["vec"] = None
    return out

# NOTE: the four helpers above are defined UNCONDITIONALLY (outside the RUN_CORE guard) because
# every later experiment cell calls them. Only the measurements below are behind the flag.
if RUN_CORE:
    # recon_cos (continuous metric -> bootstrap CI); task map should sit much lower
    rc_recon = F.cosine_similarity(map_recon(X9e).cpu(), X2e, dim=1).numpy()
    rc_task = F.cosine_similarity(((X9e.to(DEVICE)-mu9d)@task_maps[0][0]+task_maps[0][1]).cpu(), X2e, dim=1).numpy()
    arr = {"recon_cos_recon": fmt(bootstrap_ci(rc_recon)),
           "recon_cos_task": fmt(bootstrap_ci(rc_task))}

    # headline bin: UNSOLVABLE — full-answer + the 5-seed treatment live here
    if unsolv:
        arr["native_unsolv_full"] = fmt(wilson_bools(arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in unsolv], vecs=None)))
        arr["recon_unsolv_full"] = fmt(wilson_bools(full_confer(map_recon, unsolv)))
        arr["recon_unsolv_first"] = fmt(wilson_bools(first_token_confer(map_recon, unsolv)))
        seed_full = [float(np.mean(full_confer(map_task(s), unsolv))) for s in range(len(task_maps))]
        arr["task_unsolv_full_acrossseed"] = fmt(across_seed_ci(seed_full))
        arr["task_unsolv_full_per_seed"] = [round(v, 4) for v in seed_full]
        arr["task_unsolv_first_pooled"] = fmt(wilson_bools(first_token_confer(map_task(0), unsolv)))
        arr["shuffle_unsolv_first"] = fmt(wilson_bools(shuffle_confer(unsolv)))
    # sanity bin: SOLVABLE — first-token + full-answer (recon and task seed 0)
    if solv:
        arr["native_solv_full"] = fmt(wilson_bools(arith_fullanswer_correct(model_2b, L2_SINGLE, [evalp[j] for j in solv], vecs=None)))
        arr["recon_solv_first"] = fmt(wilson_bools(first_token_confer(map_recon, solv)))
        arr["recon_solv_full"] = fmt(wilson_bools(full_confer(map_recon, solv)))
        arr["task_solv_first"] = fmt(wilson_bools(first_token_confer(map_task(0), solv)))
        arr["task_solv_full"] = fmt(wilson_bools(full_confer(map_task(0), solv)))
    RESULTS["arithmetic"] = arr
    print("EXP2/3:", json.dumps(arr, indent=2))
else:
    print("core-channel eval skipped (RUN_CORE=False); map_task/first_token_confer/full_confer "
          "are still defined for the later cells.")


EXP2/3: {
  "recon_cos_recon": "0.982 [0.981, 0.983]",
  "recon_cos_task": "0.193 [0.191, 0.196]",
  "native_unsolv_full": "0.006 [0.002, 0.020]",
  "recon_unsolv_full": "0.056 [0.036, 0.084]",
  "recon_unsolv_first": "0.275 [0.231, 0.323]",
  "task_unsolv_full_acrossseed": "0.137 [0.125, 0.148]",
  "task_unsolv_full_per_seed": [
    0.1417,
    0.125,
    0.1333,
    0.15,
    0.1333
  ],
  "task_unsolv_first_pooled": "0.814 [0.770, 0.851]",
  "shuffle_unsolv_first": "0.150 [0.117, 0.191]",
  "native_solv_full": "0.213 [0.193, 0.235]",
  "recon_solv_first": "0.954 [0.942, 0.963]",
  "recon_solv_full": "0.187 [0.168, 0.208]",
  "task_solv_first": "0.927 [0.913, 0.940]",
  "task_solv_full": "0.143 [0.126, 0.162]"
}


In [24]:
# === CELL 10b: SELF-GRAFT control (hardens the faithfulness axis) ===
# Base: CELL E12 of Gemma_27Bto2B_eval_CORE.ipynb. Two checks on the UNSOLVABLE bin:
#   (1) self-graft: re-inject the RECIPIENT's OWN state at L2_SINGLE. Must equal native — it
#       proves the hook machinery adds nothing on its own (no accidental conferral from the
#       act of grafting) and that any conferral below comes from the DONOR's content.
#   (2) recon-map graft: must also ~= native, confirming the recon map lands on the recipient
#       manifold (it reconstructs, it does not transfer).
selfmap = {}
if RUN_CORE and unsolv:
    @torch.inference_mode()
    def _graft_states_confer(states, idxs):     # graft an explicit per-item state vector
        h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch); out = []
        try:
            for i in range(0, len(idxs), ARITH_BATCH):
                sub = idxs[i:i+ARITH_BATCH]
                _graft["vec"] = states[sub].to(DEVICE).float()
                ids, m = left_pad([evalp[j]["ids"] for j in sub], tokenizer.pad_token_id)
                top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
                out += [top[k] == evalp[sub[k]]["tok"] for k in range(len(sub))]
        finally:
            h.remove(); _graft["vec"] = None
        return out
    selfmap["native_unsolv_first"] = fmt(wilson_bools([two_top[j] == evalp[j]["tok"] for j in unsolv]))
    selfmap["selfgraft_unsolv_first"] = fmt(wilson_bools(_graft_states_confer(X2e, unsolv)))
    selfmap["selfgraft_unsolv_full"] = fmt(wilson_bools(arith_fullanswer_correct(
        model_2b, L2_SINGLE, [evalp[j] for j in unsolv], vecs=list(X2e[unsolv].float()))))
    selfmap["reconmap_unsolv_first"] = RESULTS.get("arithmetic", {}).get("recon_unsolv_first", "n/a")
    selfmap["taskmap_unsolv_first"] = RESULTS.get("arithmetic", {}).get("task_unsolv_first_pooled", "n/a")
    selfmap["recon_cos_recon_reference"] = RESULTS.get("arithmetic", {}).get("recon_cos_recon", "n/a")
    selfmap["_expect"] = ("selfgraft ~= native (hook is faithful); reconmap ~= native (recon "
                          "reconstructs); taskmap >> native (transcription)")
RESULTS["selfgraft_control"] = selfmap
print("self-graft control:", json.dumps(selfmap, indent=2))


self-graft control: {
  "native_unsolv_first": "0.000 [0.000, 0.011]",
  "selfgraft_unsolv_first": "0.000 [0.000, 0.011]",
  "selfgraft_unsolv_full": "0.000 [0.000, 0.011]",
  "reconmap_unsolv_first": "0.275 [0.231, 0.323]",
  "taskmap_unsolv_first": "0.814 [0.770, 0.851]",
  "recon_cos_recon_reference": "0.982 [0.981, 0.983]",
  "_expect": "selfgraft ~= native (hook is faithful); reconmap ~= native (recon reconstructs); taskmap >> native (transcription)"
}


In [25]:
# === CELL 11: EXP4 — donor-presence probe (arithmetic PRESENT vs GSM8K ABSENT) ===
# Base: CELL 11 of consolidated_eval.ipynb. This is the "present" half of the law's factorisation:
# P(donor solves in one pass) x P(extract | present). Everything here is a DONOR-side quantity and
# is therefore expected to MATCH between the 9B- and 2B-recipient notebooks (up to the graft layer,
# which is re-derived per recipient — the per-layer curve in CELL 7b is the layer-free join key).
RUN_PRESENCE = globals().get("RUN_PRESENCE", True)
if RUN_PRESENCE:
    @torch.inference_mode()
    def gsm_states_donor(rows, layer, batch=GSM_BATCH):
        out = []
        for i in range(0, len(rows), batch):
            enc = tokenizer([gsm_prompt(r["question"]) for r in rows[i:i+batch]],
                            return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
            hs = model_9b.model(enc["input_ids"], attention_mask=enc["attention_mask"],
                                output_hidden_states=True).hidden_states[layer+1]
            out.append(hs[:, -1, :].float().cpu())
        return torch.cat(out)

    # arithmetic donor probe (reuse the collected donor eval states; split train/test inside)
    ya = torch.tensor([fd(p["ans"]) for p in evalp]); half = len(evalp)//2
    arith_bools = probe_split_bools(X9e, ya)
    # GSM8K donor probe
    gtr = list(load_dataset("openai/gsm8k","main",split="train").select(range(GSM_FIT*2)))
    gte = list(load_dataset("openai/gsm8k","main",split="test").select(range(min(GSM_EVAL, 200))))
    def gy(rows): return torch.tensor([fd(gsm_extract(r["answer"])) for r in rows])
    gp = probe_first_digit(gsm_states_donor(gtr, L9_SINGLE), gy(gtr), gsm_states_donor(gte, L9_SINGLE))
    gsm_bools = (gp == gy(gte)).tolist()
    RESULTS["donor_presence"] = {
        "graft_layer_donor": L9_SINGLE,
        "arith_donor_probe": fmt(wilson_bools(arith_bools)),
        "arith_majority_baseline": majority_acc(ya[half:]),
        "gsm_donor_probe": fmt(wilson_bools(gsm_bools)),
        "gsm_majority_baseline": majority_acc(gy(gte)),
        "n_arith_test": len(arith_bools), "n_gsm_test": len(gsm_bools)}
    print("EXP4:", json.dumps(RESULTS["donor_presence"], indent=2))
else:
    print("EXP4 donor-presence probe skipped (RUN_PRESENCE=False).")


EXP4 donor-presence probe skipped (RUN_PRESENCE=False).


In [26]:
# === CELL 11b (EXP8 of the plan): DIGIT-1/2/3 probes at the DONOR graft site ===
# The informational bound. Digit 1 is the token the first-token conferral metric scores; digits 2
# and 3 say how much of the WHOLE answer is linearly present in the single grafted vector. The
# full-answer conferral can never exceed what is decodable here — that is the "bound" claim.
# Probed at the derived graft layer AND at the fixed reference layer L_DONOR_REF so the two
# notebooks always share at least one comparable donor site even if layer derivation moves.
RUN_DIGITS = globals().get("RUN_DIGITS", True)
if RUN_DIGITS:
    digit_probe_layers = sorted({L9_SINGLE, L_DONOR_REF})
    digit_probe_layers = [L for L in digit_probe_layers if 0 <= L < NL_DONOR]
    Xd_by_layer, _ = states_and_top(model_9b, digit_probe_layers, [p["ids"] for p in evalp], batch=DONOR_BATCH)
    dprobe = {}
    for L in digit_probe_layers:
        XL = Xd_by_layer[L]; per_layer = {}
        for k in (1, 2, 3):
            ki = [i for i in range(len(evalp)) if _dig(evalp[i]["ans"], k) is not None]
            if len(ki) < 60:
                per_layer[f"digit{k}"] = f"n/a (only {len(ki)} items have digit {k})"; continue
            yk = torch.tensor([_dig(evalp[i]["ans"], k) for i in ki])
            Xk = XL[ki]
            per_layer[f"digit{k}"] = {"n": len(ki),
                                      "acc": fmt(wilson_bools(probe_split_bools(Xk, yk))),
                                      "majority": majority_acc(yk[len(yk)//2:])}
        dprobe[f"donorL{L}"] = per_layer
    RESULTS["donor_digit_probes"] = {
        "layers": digit_probe_layers, "graft_layer_donor": L9_SINGLE,
        "by_layer": dprobe,
        "_note": ("full-answer conferral is bounded above by how much of the answer is decodable "
                  "here; digit1 >> digit2 >> digit3 is the signature of 'the graft carries an "
                  "answer, not a computation'.")}
    print("digit-1/2/3 donor probes:", json.dumps(RESULTS["donor_digit_probes"], indent=2))
    del Xd_by_layer
else:
    print("digit-1/2/3 donor probes skipped (RUN_DIGITS=False).")


digit-1/2/3 donor probes: {
  "layers": [
    34,
    37
  ],
  "graft_layer_donor": 34,
  "by_layer": {
    "donorL34": {
      "digit1": {
        "n": 1805,
        "acc": "0.849 [0.825, 0.871]",
        "majority": 0.298
      },
      "digit2": {
        "n": 1805,
        "acc": "0.198 [0.174, 0.225]",
        "majority": 0.127
      },
      "digit3": {
        "n": 1692,
        "acc": "0.126 [0.106, 0.151]",
        "majority": 0.118
      }
    },
    "donorL37": {
      "digit1": {
        "n": 1805,
        "acc": "0.948 [0.931, 0.961]",
        "majority": 0.298
      },
      "digit2": {
        "n": 1805,
        "acc": "0.254 [0.226, 0.283]",
        "majority": 0.127
      },
      "digit3": {
        "n": 1692,
        "acc": "0.136 [0.114, 0.161]",
        "majority": 0.118
      }
    }
  },
  "_note": "full-answer conferral is bounded above by how much of the answer is decodable here; digit1 >> digit2 >> digit3 is the signature of 'the graft carries an answer, not 

In [27]:
# === CELL 12 (EXP14): PER-EXAMPLE FREE-VECTOR CEILING (true single-site upper bound) ===
# Base: EXP14_freevec_ceiling_cell.py. Removes EVERY constraint on the bridge: for each
# unsolvable problem we optimise an UNCONSTRAINED vector injected at the graft site directly
# against the FULL gold answer SEQUENCE (teacher-forced), per example. That is the best ANY
# single-site injection can do. Then we FREE-GENERATE and score the full answer.
#   teacher-forced answer CE -> ~0   (sanity: the vector CAN encode the answer)
#   free-gen FULL-answer acc -> THE CEILING.
# Caveat for the writeup: teacher forcing gives the vector the gold prefix during training, so
# the free-gen number is a CONSERVATIVE read — it makes a LOW ceiling more credible, not less.
# Subsampled to CEIL_N items (identical in both notebooks) because the optimisation is
# fwd+bwd through the recipient CEIL_STEPS times and the 9B recipient is not cheap.
RUN_CEIL   = globals().get("RUN_CEIL", True)
CEIL_STEPS = globals().get("CEIL_STEPS", 150)
CEIL_LR    = globals().get("CEIL_LR", 5e-2)

if RUN_CEIL and unsolv:
    ceil_idx = unsolv[:CEIL_N]
    items = [evalp[i] for i in ceil_idx]
    seqs, ans_lens, prompt_lens = [], [], []
    for p in items:
        pid  = p["ids"].flatten().tolist()
        full = tokenizer(_aprompt(p["expr"]) + " " + str(p["ans"])).input_ids
        if full[:len(pid)] != pid:                      # tokenizer should be prefix-consistent
            full = pid + tokenizer(" " + str(p["ans"]), add_special_tokens=False).input_ids
        seqs.append(torch.tensor(full)); prompt_lens.append(len(pid)); ans_lens.append(len(full) - len(pid))
    Lmax, B, pad_id = max(s.numel() for s in seqs), len(seqs), tokenizer.pad_token_id
    IDS = torch.full((B, Lmax), pad_id, dtype=torch.long)
    TGT = torch.full((B, Lmax), -100, dtype=torch.long)  # ignore everywhere except answer span
    PE  = torch.zeros(B, dtype=torch.long)               # graft site = last prompt token index
    for i, s in enumerate(seqs):
        n = s.numel(); off = Lmax - n                    # left-pad => right-aligned
        IDS[i, off:] = s
        pe = off + prompt_lens[i] - 1
        PE[i] = pe
        TGT[i, pe:pe + ans_lens[i]] = s[prompt_lens[i]:]  # logits@pe.. predict answer tokens
    ATT = (IDS != pad_id).long()

    # free per-example vectors, initialised from the reconstruction-map output (on-manifold start)
    V = map_recon(X9e[ceil_idx]).detach().clone().to(DEVICE).float().requires_grad_(True)
    opt = torch.optim.Adam([V], lr=CEIL_LR)
    _ceil = {"V": None, "pe": None}
    def _ceil_hook(_m, _i, o):                           # graft V[i] at absolute position pe[i]
        h = _hid(o)
        if _ceil["V"] is None or h.shape[1] <= 1: return o
        h2 = h.clone()
        h2[torch.arange(h.shape[0], device=h.device), _ceil["pe"].to(h.device), :] = _ceil["V"].to(h.dtype)
        return _pack(o, h2)
    model_2b.requires_grad_(False)
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(_ceil_hook)
    try:
        for step in range(CEIL_STEPS):
            perm = torch.randperm(B); tot = 0.0
            for s in range(0, B, ARITH_BATCH):
                sub = perm[s:s + ARITH_BATCH]
                _ceil["V"] = V[sub]; _ceil["pe"] = PE[sub]
                logits = model_2b(IDS[sub].to(DEVICE), attention_mask=ATT[sub].to(DEVICE)).logits.float()
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                       TGT[sub].reshape(-1).to(DEVICE), ignore_index=-100)
                opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(sub)
            if step % 25 == 0 or step == CEIL_STEPS - 1:
                print(f"  ceil step {step:3d}  teacher-forced answer CE = {tot / B:.4f}", flush=True)
    finally:
        handle.remove(); _ceil["V"] = None
    model_2b.requires_grad_(True)
    Vf = V.detach()

    @torch.inference_mode()
    def _ceil_first(Vrows):
        fok, h = [], model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        try:
            for s in range(0, B, ARITH_BATCH):
                sub = list(range(s, min(s + ARITH_BATCH, B)))
                _graft["vec"] = Vrows[sub].to(DEVICE)
                ids, m = left_pad([items[j]["ids"] for j in sub], tokenizer.pad_token_id)
                top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
                fok += [top[k] == items[sub[k]]["tok"] for k in range(len(sub))]
        finally:
            h.remove(); _graft["vec"] = None
        return fok
    def _ceil_full(Vrows):
        return arith_fullanswer_correct(model_2b, L2_SINGLE, items, vecs=list(Vrows.cpu()))

    fok, full = _ceil_first(Vf), _ceil_full(Vf)
    ref = RESULTS.get("arithmetic", {})
    RESULTS["freevec_ceiling"] = {
        "n_unsolv_used":    B,
        "ceiling_first":    fmt(wilson_bools(fok)),       # ~1.0 expected (sanity)
        "ceiling_full":     fmt(wilson_bools(full)),      # THE CEILING
        "_ref_task_full":   ref.get("task_unsolv_full_acrossseed", "n/a"),
        "_ref_native_full": ref.get("native_unsolv_full", "n/a"),
    }
    print("EXP14 free-vector ceiling:", json.dumps(RESULTS["freevec_ceiling"], indent=2))
else:
    print("EXP14 skipped (RUN_CEIL=False or no unsolvable bin).")


  ceil step   0  teacher-forced answer CE = 1.5462
  ceil step  25  teacher-forced answer CE = 0.1958
  ceil step  50  teacher-forced answer CE = 0.0200
  ceil step  75  teacher-forced answer CE = 0.0051
  ceil step 100  teacher-forced answer CE = 0.0025
  ceil step 125  teacher-forced answer CE = 0.0015
  ceil step 149  teacher-forced answer CE = 0.0010
EXP14 free-vector ceiling: {
  "n_unsolv_used": 200,
  "ceiling_first": "1.000 [0.981, 1.000]",
  "ceiling_full": "0.985 [0.957, 0.995]",
  "_ref_task_full": "0.137 [0.125, 0.148]",
  "_ref_native_full": "0.006 [0.002, 0.020]"
}


In [28]:
# === CELL 13 (EXP6): TRANSCRIPTION PROBE — did the MAP write the answer in? ===
# Base: CELL 14 of consolidated_eval.ipynb. The task graft REPLACES the recipient's L2_SINGLE
# state with the map output, so "the recipient's state after the graft" at that layer IS the
# stitched vector. Probe the answer's first digit from: native recipient state (no graft),
# reconstruction-map output, task-map output (the stitched vector), and the DONOR graft state
# (reference ceiling). On UNSOLVABLE problems the native state should NOT carry the answer,
# while the stitched vector should — at ~donor level. That gap is transcription.
# Cheap: no model forwards, just linear probes on states already in memory.
RUN_TRANSCRIBE = globals().get("RUN_TRANSCRIBE", True)
if RUN_TRANSCRIBE:
    pidx = unsolv if len(unsolv) >= 60 else list(range(len(evalp)))
    y_tc = torch.tensor([fd(evalp[i]["ans"]) for i in pidx])
    X2_sub, X9_sub = X2e[pidx], X9e[pidx]
    W0, b0 = task_maps[0]
    task_out = ((X9_sub.to(DEVICE)-mu9d)@W0 + b0).detach().cpu()
    recon_out = map_recon(X9_sub).detach().cpu()
    def _probe_acc(X): return wilson_bools(probe_split_bools(X, y_tc))
    RESULTS["transcription_confirm"] = {
        "n": len(pidx),
        "majority_baseline": majority_acc(y_tc[len(y_tc)//2:]),
        f"native_recipient_L{L2_SINGLE}": fmt(_probe_acc(X2_sub)),
        "recon_map_output": fmt(_probe_acc(recon_out)),
        "task_map_output_STITCHED_VECTOR": fmt(_probe_acc(task_out)),
        f"donor_graft_state_L{L9_SINGLE}": fmt(_probe_acc(X9_sub)),
    }
    print("EXP6 transcription probe:", json.dumps(RESULTS["transcription_confirm"], indent=2))
else:
    print("EXP6 transcription probe skipped (RUN_TRANSCRIBE=False).")


EXP6 transcription probe: {
  "n": 360,
  "majority_baseline": 0.183,
  "native_recipient_L20": "0.639 [0.566, 0.705]",
  "recon_map_output": "0.611 [0.538, 0.679]",
  "task_map_output_STITCHED_VECTOR": "0.772 [0.706, 0.827]",
  "donor_graft_state_L34": "0.706 [0.635, 0.767]"
}


In [29]:
# === CELL 13b (HEADLINE TEST): KL + MSE-anchor sweep - the manifold/conferral trade-off ===
# THE question for this notebook: does an instruction-tuned recipient let an ON-manifold map
# confer more than the ~0.19 base-model recon floor? lambda=0 is pure KL (free to leave the
# manifold); large lambda pins the graft onto the recipient's own states (recon-like).
# Compare (recon_cos, conferral) here against the v1 base numbers in the header markdown.
# Answers two things in ONE sweep over the anchor weight lambda:
#   (a) PURE KL  (lambda = 0): can a linear map carry the DONOR'S FULL OUTPUT DISTRIBUTION
#       (not just the argmax answer token that the CE task map targets) into the recipient?
#   (b) KL + MSE (lambda > 0): does any conferral SURVIVE pinning the graft onto the
#       recipient's own activation manifold? The MSE term is exactly the recon objective
#       ( ||graft - 2B_own_state||^2 ). Sweeping lambda 0 -> large traces free/off-manifold
#       -> recon/on-manifold. If conferral exists ONLY at low lambda (off-manifold), the
#       "transfer" is answer-injection — same conclusion EXP6 reaches for the CE task map.
#       recon_cos reports where on that axis each trained map actually landed.
# Reuses (must already be in memory): Wr, mu2, mu9d, X9t, X2t, X9e, X2e, train, evalp,
#   unsolv, first_token_confer, full_confer, probe_first_digit, fd, patch_vec_batch, _graft,
#   left_pad, wilson_bools, fmt, bootstrap_ci.  PREREQUISITE CELLS, in order: 2, 2b, 4, 5, 5b,
#   6, 8, 9 (mu9d + task_maps), 10 (first_token_confer/full_confer, defined unconditionally),
#   and 13 (populates RESULTS["transcription_confirm"], used only for the _ref_* rows).

RUN_DISTILL = globals().get("RUN_DISTILL", True)
LAMBDAS     = globals().get("LAMBDAS", [0.0, 0.1, 1.0, 10.0])   # 0.0 = pure KL; rest add the anchor
KL_EPOCHS   = globals().get("KL_EPOCHS", TASK_EPOCHS)
KL_TEMP     = globals().get("KL_TEMP", 1.0)                      # softmax temperature for distillation

if RUN_DISTILL:
    # ---- teacher: 9B native last-token logits on the SAME prompts, cached once (CPU, fp16) ----
    # NOTE on memory: cache is [N_train, vocab(~256k)] fp16 ~= 0.5 GB per 1k train items, on CPU.
    # If CPU RAM is tight, move this computation inline into the loop (recompute per batch).
    @torch.inference_mode()
    def _teacher_logits(items):
        out = []
        for s in range(0, len(items), ARITH_BATCH):
            sub = items[s:s + ARITH_BATCH]
            ids, m = left_pad([p["ids"] for p in sub], tokenizer.pad_token_id)
            lg = model_9b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
            out.append(lg.half().cpu())
        return torch.cat(out, 0)
    print(f"caching 9B teacher logits for {len(train)} train items ...")
    T_train = _teacher_logits(train)                       # clean 9B (no graft active here)
    print(f"  teacher cache: {tuple(T_train.shape)} fp16  (~{T_train.numel()*2/1e9:.2f} GB CPU)")

    def train_distill_map(lam):
        torch.manual_seed(0)
        W = Wr.clone().to(DEVICE).requires_grad_(True)     # warm-start from the ridge solution
        b = mu2.clone().to(DEVICE).requires_grad_(True)
        opt = torch.optim.Adam([W, b], lr=1e-3)
        model_2b.requires_grad_(False)
        handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        idx = list(range(len(train)))
        last_kl = 0.0
        try:
            for ep in range(KL_EPOCHS):
                random.Random(ep).shuffle(idx)
                for s in range(0, len(idx), ARITH_BATCH):
                    sub = idx[s:s + ARITH_BATCH]
                    x9 = X9t[sub].to(DEVICE)
                    graft = (x9 - mu9d) @ W + b
                    _graft["vec"] = graft                  # patched into 2B at L2_SINGLE
                    ids, m = left_pad([train[k]["ids"] for k in sub], tokenizer.pad_token_id)
                    student = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    teacher = T_train[sub].to(DEVICE).float()
                    # forward KL( teacher || student ): student must cover the donor's distribution
                    kl = F.kl_div(F.log_softmax(student / KL_TEMP, -1),
                                  F.softmax(teacher / KL_TEMP, -1),
                                  reduction="batchmean") * (KL_TEMP ** 2)
                    loss = kl
                    if lam > 0:                            # anchor onto the recipient's OWN state
                        loss = loss + lam * F.mse_loss(graft, X2t[sub].to(DEVICE).float())
                    opt.zero_grad(); loss.backward(); opt.step()
                    last_kl = float(kl.item())
        finally:
            handle.remove(); _graft["vec"] = None
        model_2b.requires_grad_(True)
        return W.detach(), b.detach(), last_kl

    # ---- transcription probe (EXP6 logic): probe the MAP OUTPUT for the first answer digit ----
    pidx = unsolv if len(unsolv) >= 60 else list(range(len(evalp)))
    y_tc = torch.tensor([fd(evalp[i]["ans"]) for i in pidx]); h_tc = len(pidx) // 2
    def _probe_map_output(W, b):
        out = ((X9e[pidx].to(DEVICE) - mu9d) @ W + b).detach().cpu()
        pred = probe_first_digit(out[:h_tc], y_tc[:h_tc], out[h_tc:], y_tc[h_tc:])
        return fmt(wilson_bools((pred == y_tc[h_tc:]).tolist()))

    sweep = {}
    for lam in LAMBDAS:
        W, b, klfin = train_distill_map(lam)
        mapfn = (lambda W, b: (lambda x9: (x9.to(DEVICE) - mu9d) @ W + b))(W, b)   # bind by value
        rc = F.cosine_similarity(mapfn(X9e).cpu(), X2e, dim=1).numpy()
        row = {
            "final_KL":         round(klfin, 4),
            "recon_cos":        fmt(bootstrap_ci(rc)),                      # on-/off-manifold
            "unsolv_first":     fmt(wilson_bools(first_token_confer(mapfn, unsolv))) if unsolv else "n/a",
            "map_output_probe": _probe_map_output(W, b),                    # > native => transcription
        }
        # full-answer uses the slow generation path: only the two endpoints (pure-KL + strongest anchor)
        if unsolv and (lam == LAMBDAS[0] or lam == LAMBDAS[-1]):
            row["unsolv_full"] = fmt(wilson_bools(full_confer(mapfn, unsolv)))
        tag = "pure_KL" if lam == 0 else f"KL+{lam}xMSE"
        sweep[tag] = row
        print(f"  lambda={lam:<6} recon_cos~{rc.mean():.3f}  {row}")

    # reference rows (already computed in EXP2/3 + EXP6) surfaced for one-glance comparison
    A = RESULTS.get("arithmetic", {}); T = RESULTS.get("transcription_confirm", {})
    sweep["_ref_recon_map"]   = {"recon_cos": A.get("recon_cos_recon"),
                                 "unsolv_first": A.get("recon_unsolv_first"),
                                 "map_output_probe": T.get("recon_map_output")}
    sweep["_ref_CE_task_map"] = {"recon_cos": A.get("recon_cos_task"),
                                 "unsolv_first": A.get("task_unsolv_first_pooled"),
                                 "map_output_probe": T.get("task_map_output_STITCHED_VECTOR")}
    sweep["_ref_native_donor"] = {"native_2B": T.get(f"native_recipient_L{L2_SINGLE}"),
                                  "donor_9B": T.get(f"donor_graft_state_L{L9_SINGLE}")}
    RESULTS["distill_kl_sweep"] = sweep
    print("EXP13 distillation KL sweep:", json.dumps(sweep, indent=2))
else:
    print("EXP13 skipped (set RUN_DISTILL=True).")


caching 9B teacher logits for 3000 train items ...
  teacher cache: (3000, 256000) fp16  (~1.54 GB CPU)
  lambda=0.0    recon_cos~0.232  {'final_KL': 0.0106, 'recon_cos': '0.232 [0.230, 0.235]', 'unsolv_first': '0.903 [0.868, 0.929]', 'map_output_probe': '0.744 [0.676, 0.803]', 'unsolv_full': '0.172 [0.137, 0.215]'}
  lambda=0.1    recon_cos~0.628  {'final_KL': 0.1667, 'recon_cos': '0.628 [0.624, 0.632]', 'unsolv_first': '0.689 [0.639, 0.735]', 'map_output_probe': '0.683 [0.612, 0.747]'}
  lambda=1.0    recon_cos~0.897  {'final_KL': 0.1664, 'recon_cos': '0.897 [0.895, 0.899]', 'unsolv_first': '0.575 [0.523, 0.625]', 'map_output_probe': '0.667 [0.595, 0.731]'}
  lambda=10.0   recon_cos~0.942  {'final_KL': 2.8231, 'recon_cos': '0.942 [0.941, 0.943]', 'unsolv_first': '0.411 [0.361, 0.463]', 'map_output_probe': '0.628 [0.555, 0.695]', 'unsolv_full': '0.100 [0.073, 0.135]'}
EXP13 distillation KL sweep: {
  "pure_KL": {
    "final_KL": 0.0106,
    "recon_cos": "0.232 [0.230, 0.235]",
    "un

In [30]:
# === CELL 14 (EXP8): NONLINEAR (MLP) task map — is transcription map-agnostic? ===
# Base: CELL 16 of consolidated_eval.ipynb. Same task-supervised objective, but a nonlinear
# (residual MLP) map on top of the linear recon map. If even a nonlinear map cannot push
# conferral ABOVE the donor-probe ceiling, transcription is a property of the DONOR's content,
# not an artifact of using a linear bridge. Warm-started at the recon map (output zero-init).
RUN_MLP = globals().get("RUN_MLP", True)
if RUN_MLP:
    torch.manual_seed(0)
    d9, d2 = X9t.shape[1], X2t.shape[1]
    mlp = nn.Sequential(nn.Linear(d9, MLP_HIDDEN), nn.GELU(), nn.Linear(MLP_HIDDEN, d2)).to(DEVICE)
    nn.init.normal_(mlp[2].weight, std=1e-3); nn.init.zeros_(mlp[2].bias)  # ~recon map, trainable everywhere
    opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
    model_2b.requires_grad_(False)
    handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
    idx = list(range(len(train)))
    try:
        for ep in range(TASK_EPOCHS):
            random.Random(ep).shuffle(idx)
            for s in range(0, len(idx), ARITH_BATCH):
                sub = idx[s:s+ARITH_BATCH]
                x9b = X9t[sub].to(DEVICE)
                _graft["vec"] = apply_map(x9b, recon_map) + mlp(x9b)
                ids, m = left_pad([train[k]["ids"] for k in sub], tokenizer.pad_token_id)
                lg = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                tgt = torch.tensor([train[k]["tok"] for k in sub], device=DEVICE)
                loss = F.cross_entropy(lg, tgt); opt.zero_grad(); loss.backward(); opt.step()
    finally:
        handle.remove(); _graft["vec"] = None
    model_2b.requires_grad_(True); mlp.eval()
    print(f"  MLP final batch CE {loss.item():.3f}")
    def map_mlp(x9):
        with torch.no_grad():
            x9 = x9.to(DEVICE)
            return apply_map(x9, recon_map) + mlp(x9)
    nlmap = {"mlp_hidden": MLP_HIDDEN,
             "mlp_params": int(sum(p.numel() for p in mlp.parameters())),
             "linear_map_params": int(task_maps[0][0].numel() + task_maps[0][1].numel())}
    if unsolv:
        nlmap["mlp_unsolv_first"] = fmt(wilson_bools(first_token_confer(map_mlp, unsolv)))
        nlmap["mlp_unsolv_full"] = fmt(wilson_bools(full_confer(map_mlp, unsolv)))
    nlmap["donor_probe_ceiling"] = RESULTS.get("donor_presence", {}).get("arith_donor_probe", "n/a")
    nlmap["linear_task_first_for_reference"] = RESULTS.get("arithmetic", {}).get("task_unsolv_first_pooled", "n/a")
    nlmap["linear_task_full_for_reference"] = RESULTS.get("arithmetic", {}).get("task_unsolv_full_acrossseed", "n/a")
    RESULTS["nonlinear_task_map"] = nlmap
    print("EXP8 nonlinear task map:", json.dumps(nlmap, indent=2))
else:
    print("EXP8 skipped (RUN_MLP=False).")


  MLP final batch CE 0.041
EXP8 nonlinear task map: {
  "mlp_hidden": 1024,
  "mlp_params": 6032640,
  "linear_map_params": 8259840,
  "mlp_unsolv_first": "0.733 [0.685, 0.776]",
  "mlp_unsolv_full": "0.147 [0.114, 0.188]",
  "donor_probe_ceiling": "n/a",
  "linear_task_first_for_reference": "0.814 [0.770, 0.851]",
  "linear_task_full_for_reference": "0.137 [0.125, 0.148]"
}


In [31]:
# === CELL 15 (EXP10): LOW-RANK TRUNCATION SWEEP of the task map (capacity control) ===
# Base: the `lowrank_sweep` logic of CELL E10 in Gemma_27Bto2B_eval_CORE.ipynb. If a rank-r map
# transcribes about as well as the full-rank map, the transferred signal is LOW-DIMENSIONAL —
# consistent with "an answer" (a few bits) rather than rich distributed computation. Together
# with the MLP cell above this brackets bridge capacity from BOTH sides: more capacity (MLP)
# doesn't help, and far less capacity (rank 8-32) doesn't hurt.
# No retraining: SVD-truncate the already-trained task map W (seed 0).
RUN_LOWRANK = globals().get("RUN_LOWRANK", True)
if RUN_LOWRANK and unsolv:
    Wt, bt = task_maps[0]                                            # (d_donor x d_recip), (d_recip,)
    U, S, Vh = torch.linalg.svd(Wt.float(), full_matrices=False)     # Wt = U diag(S) Vh
    def map_lowrank(r):
        Wr_ = ((U[:, :r] * S[:r]) @ Vh[:r, :]).to(DEVICE)            # rank-r truncation of Wt
        btd = bt.to(DEVICE)
        return lambda x9: (x9.to(DEVICE) - mu9d) @ Wr_ + btd
    full_dim = int(min(Wt.shape))
    ranks = [r for r in LOWRANK_RANKS if r < full_dim] + [full_dim]
    lowrank = {"full_rank_dim": full_dim, "d_recipient": int(X2t.shape[1])}
    for r in ranks:
        tag = "full" if r == full_dim else str(r)
        lowrank[f"r{tag}_unsolv_first"] = fmt(wilson_bools(first_token_confer(map_lowrank(r), unsolv)))
        print(f"  rank {tag:>5s}: first-token conferral {lowrank[f'r{tag}_unsolv_first']}", flush=True)
    # full-answer at a few informative ranks only (generation is the expensive part)
    for r in [rr for rr in LOWRANK_FULL_RANKS if rr < full_dim] + [full_dim]:
        tag = "full" if r == full_dim else str(r)
        lowrank[f"r{tag}_unsolv_full"] = fmt(wilson_bools(full_confer(map_lowrank(r), unsolv)))
    e = (S.pow(2).cumsum(0) / S.pow(2).sum()).tolist()
    lowrank["sv_energy_at_r"] = {f"r{r}": round(e[min(r-1, len(e)-1)], 3) for r in ranks}
    lowrank["task_first_reference"] = RESULTS.get("arithmetic", {}).get("task_unsolv_first_pooled", "n/a")
    RESULTS["lowrank_sweep"] = lowrank
    print("EXP10 low-rank sweep:", json.dumps(lowrank, indent=2))
else:
    print("EXP10 skipped (RUN_LOWRANK=False or no unsolvable bin).")


  rank     1: first-token conferral 0.264 [0.221, 0.312]
  rank     2: first-token conferral 0.369 [0.321, 0.420]
  rank     4: first-token conferral 0.417 [0.367, 0.468]
  rank     8: first-token conferral 0.525 [0.473, 0.576]
  rank    16: first-token conferral 0.758 [0.712, 0.800]
  rank    32: first-token conferral 0.758 [0.712, 0.800]
  rank    64: first-token conferral 0.806 [0.762, 0.843]
  rank   128: first-token conferral 0.822 [0.779, 0.858]
  rank  full: first-token conferral 0.811 [0.767, 0.848]
EXP10 low-rank sweep: {
  "full_rank_dim": 2304,
  "d_recipient": 2304,
  "r1_unsolv_first": "0.264 [0.221, 0.312]",
  "r2_unsolv_first": "0.369 [0.321, 0.420]",
  "r4_unsolv_first": "0.417 [0.367, 0.468]",
  "r8_unsolv_first": "0.525 [0.473, 0.576]",
  "r16_unsolv_first": "0.758 [0.712, 0.800]",
  "r32_unsolv_first": "0.758 [0.712, 0.800]",
  "r64_unsolv_first": "0.806 [0.762, 0.843]",
  "r128_unsolv_first": "0.822 [0.779, 0.858]",
  "rfull_unsolv_first": "0.811 [0.767, 0.848]",
  

In [32]:
# === CELL 16 (EXP17 + G1-INLP): ANSWER-SUBSPACE ERASURE with MATCHED-RANK RANDOM CONTROL ===
# Base: EXP17_answer_ablation_cell.py + the INLP half of CELL G1 in Followup2_Gemma.ipynb.
# The transcription probe (CELL 13) is CORRELATIONAL. This makes it CAUSAL: iteratively remove
# answer-carrying directions from the STITCHED vector (INLP), re-graft, and watch conferral fall.
# The matched-rank RANDOM subspace is the control — deleting the same NUMBER of directions at
# random must NOT collapse conferral, or the effect is just "we deleted a lot of the vector".
#
# *** HEADLINE METRIC FOR THE CROSS-RECIPIENT COMPARISON ***
# collapse_rank_over_dmodel = (rank at which conferral falls below COLLAPSE_FRAC x baseline)
#                             / recipient d_model
# With the donor held fixed, a LARGER normalised collapse rank in the bigger recipient means the
# bigger recipient spreads the transcribed answer over proportionally more directions, i.e. it
# stores the answer more REDUNDANTLY. That is the "do larger models store the answer more
# redundantly?" question, made quantitative and comparable across different d_model.
RUN_INLP = globals().get("RUN_INLP", True)

if RUN_INLP and unsolv:
    base = map_task(0)                                  # the exact vector CELL 10 grafts
    d_recip = int(X2t.shape[1])
    # Define the answer subspace WITHOUT touching the eval bin we score on: fit the readout on
    # the task-map output of the TRAIN items (disjoint from unsolv).
    Ttr = base(X9t).detach()
    ytr = torch.tensor([fd(p["ans"]) for p in train], device=DEVICE)
    def _proj_out(X, Q): return X if Q is None else X - (X @ Q) @ Q.T
    def _erased(Qs):
        def f(x9):
            v = base(x9)
            return v - (v @ Qs) @ Qs.T
        return f
    def _rand_Q(r, seed=0):
        g = torch.Generator(device="cpu").manual_seed(seed)
        M = torch.randn(d_recip, r, generator=g).to(DEVICE)
        return torch.linalg.qr(M).Q[:, :r]

    baseline_bools = first_token_confer(base, unsolv)
    baseline = float(np.mean(baseline_bools))
    native_first = float(np.mean([two_top[j] == evalp[j]["tok"] for j in unsolv]))
    print(f"  INLP baseline (un-erased) first-token conferral = {baseline:.3f}; "
          f"native floor = {native_first:.3f}")

    Q = None; curve = []
    collapse_rank = None; collapse_rank_strict = None
    for rnd in range(INLP_ROUNDS):
        Pw, _mu = fit_probe_w(_proj_out(Ttr, Q), ytr)
        Pc = Pw - Pw.mean(1, keepdim=True)               # drop softmax's shift-invariant direction
        D = Pc if Q is None else Pc - Q @ (Q.T @ Pc)
        Un, Sn, _ = torch.linalg.svd(D, full_matrices=False)
        keepd = Un[:, Sn > Sn.max() * 1e-3]
        if keepd.shape[1] == 0:
            print(f"  INLP converged at round {rnd} (no discriminative directions left)"); break
        Q = keepd if Q is None else torch.linalg.qr(torch.cat([Q, keepd], 1)).Q
        r = int(Q.shape[1])
        if (rnd + 1) % INLP_CHECK_EVERY == 0 or rnd == INLP_ROUNDS - 1:
            Qr = _rand_Q(r, seed=0)
            acc = float(np.mean(first_token_confer(_erased(Q), unsolv)))
            accr = float(np.mean(first_token_confer(_erased(Qr), unsolv)))
            curve.append({"rank": r, "rank_over_dmodel": round(r / d_recip, 4),
                          "answer_erased_first": round(acc, 4),
                          "random_erased_first": round(accr, 4)})
            print(f"  rank {r:4d} ({r/d_recip:.3f} of d_model): answer-erased={acc:.3f}  "
                  f"random-erased={accr:.3f}", flush=True)
            if collapse_rank is None and acc <= COLLAPSE_FRAC * baseline: collapse_rank = r
            if collapse_rank_strict is None and acc <= native_first + 0.02: collapse_rank_strict = r
            if collapse_rank_strict is not None: break     # fully collapsed; no need to erase further
    assert Q is not None, ("INLP found no discriminative answer directions at round 0 — the task "
                           "map output carries no linearly decodable first digit at all. Check "
                           "CELL 13 (transcription probe) before interpreting anything downstream.")
    r_final = int(Q.shape[1])
    Qr_final = _rand_Q(r_final, seed=0)
    out = {
        "d_recipient": d_recip,
        "inlp_rounds_run": len(curve) * INLP_CHECK_EVERY,
        "baseline_task_first": fmt(wilson_bools(baseline_bools)),
        "native_floor_first": round(native_first, 4),
        "final_rank": r_final,
        "final_rank_over_dmodel": round(r_final / d_recip, 4),
        "answer_erased_first_at_final_rank": fmt(wilson_bools(first_token_confer(_erased(Q), unsolv))),
        "random_erased_first_at_final_rank": fmt(wilson_bools(first_token_confer(_erased(Qr_final), unsolv))),
        "collapse_frac_threshold": COLLAPSE_FRAC,
        "collapse_rank": collapse_rank,
        "collapse_rank_over_dmodel": (round(collapse_rank / d_recip, 4) if collapse_rank else None),
        "collapse_rank_to_native_floor": collapse_rank_strict,
        "collapse_rank_to_native_floor_over_dmodel": (round(collapse_rank_strict / d_recip, 4)
                                                      if collapse_rank_strict else None),
        "curve": curve,
        "_ref_shuffle_first": RESULTS.get("arithmetic", {}).get("shuffle_unsolv_first", "n/a"),
        "_headline": ("collapse_rank_over_dmodel is the redundancy metric: bigger => the recipient "
                      "spreads the transcribed answer over proportionally more directions."),
    }
    # single-shot (rank<=9) ablation too — the original EXP17 measurement, for continuity
    Pw1, _ = fit_probe_w(Ttr, ytr)
    Pc1 = Pw1 - Pw1.mean(1, keepdim=True)
    U1, S1, _ = torch.linalg.svd(Pc1, full_matrices=False)
    k1 = int((S1 > S1.max() * 1e-3).sum().item())
    Q1 = U1[:, :k1]; Q1r = _rand_Q(k1, seed=1)
    out["oneshot_rank_k"] = k1
    out["oneshot_answer_ablated_first"] = fmt(wilson_bools(first_token_confer(_erased(Q1), unsolv)))
    out["oneshot_random_ablated_first"] = fmt(wilson_bools(first_token_confer(_erased(Q1r), unsolv)))
    if ABLATE_FULL:
        out["oneshot_task_full"] = RESULTS.get("arithmetic", {}).get("task_unsolv_full_acrossseed", "n/a")
        out["oneshot_answer_ablated_full"] = fmt(wilson_bools(full_confer(_erased(Q1), unsolv)))
    RESULTS["answer_subspace_erasure"] = out
    print("EXP17/INLP answer-subspace erasure:", json.dumps(out, indent=2))
else:
    print("EXP17/INLP skipped (RUN_INLP=False or no unsolvable bin).")


  INLP baseline (un-erased) first-token conferral = 0.814; native floor = 0.000
  rank   36 (0.016 of d_model): answer-erased=0.681  random-erased=0.811
  rank   72 (0.031 of d_model): answer-erased=0.561  random-erased=0.811
  rank  108 (0.047 of d_model): answer-erased=0.425  random-erased=0.819
  rank  144 (0.062 of d_model): answer-erased=0.308  random-erased=0.800
  rank  180 (0.078 of d_model): answer-erased=0.267  random-erased=0.811
  rank  216 (0.094 of d_model): answer-erased=0.231  random-erased=0.808
  rank  252 (0.109 of d_model): answer-erased=0.231  random-erased=0.803
  rank  288 (0.125 of d_model): answer-erased=0.203  random-erased=0.783
  rank  324 (0.141 of d_model): answer-erased=0.181  random-erased=0.783
  rank  360 (0.156 of d_model): answer-erased=0.189  random-erased=0.811
  rank  396 (0.172 of d_model): answer-erased=0.200  random-erased=0.786
  rank  432 (0.188 of d_model): answer-erased=0.186  random-erased=0.794
  rank  468 (0.203 of d_model): answer-erase

In [33]:
# === CELL 17 (EXP21): RECIPIENT-SIDE SAE FEATURE DELTA — what does the stitch CHANGE? ===
# Base: EXP21_recipient_feature_delta_cell.py. Decode the RECIPIENT residual with a GemmaScope
# residual SAE at the graft layer and compare the STITCHED state (task-map output) against the
# NATIVE state. The per-feature delta is exactly what the stitch turns on/off inside the recipient.
# Prediction: the stitch injects the recipient's ANSWER features (the ones it natively uses to
# encode answers on SOLVABLE problems) and little else. The reconstruction map is the contrast:
# it should move almost nothing. Reported: (a) added-feature answer-SELECTIVITY RATIO and
# (b) TOTAL FEATURE MOVEMENT (L1) — both requested for the cross-recipient comparison.
# Both recipients have full GemmaScope residual coverage; SAE_RELEASE/SAE_LAYER are set in CELL 2b.
RUN_RDELTA = globals().get("RUN_RDELTA", True)
TOPK_DELTA = globals().get("TOPK_DELTA", 30)

if RUN_RDELTA and unsolv:
    try:
        from sae_lens import SAE
        sae2 = None; sae_id_used = None
        for cand_layer in [L2_SINGLE] + [L2_SINGLE + d for d in (-1, 1, -2, 2, -3, 3)]:
            if not (0 <= cand_layer < NL_RECIP): continue
            sid = SAE_ID_TEMPLATE.format(layer=cand_layer)
            try:
                s = SAE.from_pretrained(SAE_RELEASE, sid, device=str(DEVICE))
                sae2 = s[0] if isinstance(s, tuple) else s
                sae_id_used = sid
                if cand_layer != L2_SINGLE:
                    print(f"  WARNING: no SAE at graft layer {L2_SINGLE}; using nearest layer {cand_layer}")
                break
            except Exception as _e:
                continue
        if sae2 is None and SAE_RELEASE != SAE_RELEASE_BASE:
            print("!" * 78)
            print("!! No IT SAE found (" + SAE_RELEASE + "). Falling back to the BASE-model SAE.")
            print("!! It was trained on gemma-2-2b (base), NOT the instruction-tuned recipient,")
            print("!! so this feature attribution is APPROXIMATE. Report it as such.")
            print("!" * 78)
            RESULTS.setdefault("_caveats", []).append(
                "recipient SAE fell back to base-model GemmaScope; IT attribution approximate")
            SAE_RELEASE = SAE_RELEASE_BASE
            for cand_layer in [L2_SINGLE] + [L2_SINGLE + d for d in (-1, 1, -2, 2, -3, 3)]:
                if not (0 <= cand_layer < NL_RECIP): continue
                sid = SAE_ID_TEMPLATE.format(layer=cand_layer)
                try:
                    s = SAE.from_pretrained(SAE_RELEASE, sid, device=str(DEVICE))
                    sae2, sae_id_used = (s[0] if isinstance(s, tuple) else s), sid
                    break
                except Exception:
                    continue
        assert sae2 is not None, f"no GemmaScope SAE found near layer {L2_SINGLE} in {SAE_RELEASE}"
        NFEAT = sae2.W_dec.shape[0]
        print(f"loaded recipient SAE {SAE_RELEASE}/{sae_id_used}: {NFEAT} features")

        @torch.inference_mode()
        def _enc_sum(vecs2d):                               # streaming feature-activation sum over rows
            tot = torch.zeros(NFEAT, device=DEVICE); n = 0
            for s in range(0, vecs2d.shape[0], ARITH_BATCH):
                a = sae2.encode(vecs2d[s:s + ARITH_BATCH].to(DEVICE).float())
                tot += a.sum(0); n += a.shape[0]
            return tot, n

        task_stitch  = (X9e[unsolv].to(DEVICE) - mu9d) @ task_maps[0][0] + task_maps[0][1]
        recon_stitch = map_recon(X9e[unsolv])
        s_task,  n1 = _enc_sum(task_stitch)
        s_recon, _  = _enc_sum(recon_stitch)
        s_nat,   _  = _enc_sum(X2e[unsolv])
        delta_task  = (s_task  - s_nat) / max(n1, 1)         # [F] stitch(task)  - native
        delta_recon = (s_recon - s_nat) / max(n1, 1)         # [F] stitch(recon) - native (contrast)

        # ANSWER features defined INTRINSICALLY: recipient features whose NATIVE activation on
        # SOLVABLE problems tracks the answer's first digit (eta^2) — independent of the stitch.
        ref_idx = solv if (solv and len(solv) >= 40) else list(range(len(evalp)))
        y = torch.tensor([fd(evalp[i]["ans"]) for i in ref_idx], device=DEVICE)
        @torch.inference_mode()
        def _eta2(vecs2d, labels):                          # per-feature answer-selectivity (eta^2 in [0,1])
            gsum = torch.zeros(10, NFEAT, device=DEVICE); gcnt = torch.zeros(10, device=DEVICE)
            tot = torch.zeros(NFEAT, device=DEVICE); tot2 = torch.zeros(NFEAT, device=DEVICE); n = 0
            for s in range(0, vecs2d.shape[0], ARITH_BATCH):
                a = sae2.encode(vecs2d[s:s + ARITH_BATCH].to(DEVICE).float()); lb = labels[s:s + a.shape[0]]
                gsum.index_add_(0, lb, a); gcnt.index_add_(0, lb, torch.ones_like(lb, dtype=torch.float))
                tot += a.sum(0); tot2 += (a * a).sum(0); n += a.shape[0]
            gmean = gsum / gcnt.clamp_min(1).unsqueeze(1); grand = tot / max(n, 1)
            between = (gcnt.unsqueeze(1) * (gmean - grand) ** 2).sum(0) / max(n, 1)
            total_var = (tot2 / max(n, 1) - grand ** 2).clamp_min(1e-8)
            return between / total_var
        sel = _eta2(X2e[ref_idx], y)

        top_add  = torch.topk(delta_task, TOPK_DELTA).indices
        top_rem  = torch.topk(-delta_task, TOPK_DELTA).indices
        base_sel = sel.mean().item()
        l1_task, l1_recon = delta_task.abs().sum().item(), delta_recon.abs().sum().item()
        out = {
            "sae": f"{SAE_RELEASE}/{sae_id_used}", "n_features": int(NFEAT),
            "n_unsolv": int(len(unsolv)), "d_recipient": int(X2t.shape[1]),
            # --- TOTAL FEATURE MOVEMENT ---
            "task_stitch_L1_change":  round(l1_task, 2),
            "recon_stitch_L1_change": round(l1_recon, 2),
            "task_over_recon_L1_ratio": round(l1_task / max(l1_recon, 1e-8), 2),
            "task_L1_per_feature": round(l1_task / NFEAT, 6),
            "n_features_moved_gt_0p01": int((delta_task.abs() > 0.01).sum().item()),
            # --- ADDED-FEATURE ANSWER-SELECTIVITY RATIO ---
            "answer_selectivity_added_features": round(sel[top_add].mean().item(), 4),
            "answer_selectivity_removed_features": round(sel[top_rem].mean().item(), 4),
            "answer_selectivity_all_features":   round(base_sel, 4),
            "added_over_baseline_ratio": round(sel[top_add].mean().item() / max(base_sel, 1e-8), 2),
            "top_added_feature_ids":   top_add.tolist()[:15],
            "top_removed_feature_ids": top_rem.tolist()[:15],
            "interpretation": ("added-feature selectivity >> baseline AND task L1 >> recon L1 => the "
                               "stitch injects the recipient's ANSWER features, not generic computation."),
        }
        RESULTS["recipient_feature_delta"] = out
        print("EXP21 recipient-side feature delta:", json.dumps(out, indent=2))
        del sae2
        import gc; gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        RESULTS["recipient_feature_delta"] = {"error": repr(e), "sae_release": SAE_RELEASE}
        print("EXP21 needs a GemmaScope residual SAE for this recipient. Fix SAE_RELEASE/SAE_ID_TEMPLATE or skip.")
        print("  error:", repr(e))
else:
    print("EXP21 skipped (RUN_RDELTA=False or no unsolvable bin).")


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!! No IT SAE found (gemma-scope-2b-it-res-canonical). Falling back to the BASE-model SAE.
!! It was trained on gemma-2-2b (base), NOT the instruction-tuned recipient,
!! so this feature attribution is APPROXIMATE. Report it as such.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


layer_20/width_16k/average_l0_71/params.(…):   0%|          | 0.00/302M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/sae_lens/saes/sae.py:249: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


loaded recipient SAE gemma-scope-2b-pt-res-canonical/layer_20/width_16k/canonical: 16384 features
EXP21 recipient-side feature delta: {
  "sae": "gemma-scope-2b-pt-res-canonical/layer_20/width_16k/canonical",
  "n_features": 16384,
  "n_unsolv": 360,
  "d_recipient": 2304,
  "task_stitch_L1_change": 232821.59,
  "recon_stitch_L1_change": 92.15,
  "task_over_recon_L1_ratio": 2526.59,
  "task_L1_per_feature": 14.210302,
  "n_features_moved_gt_0p01": 16384,
  "answer_selectivity_added_features": 0.023,
  "answer_selectivity_removed_features": 0.0098,
  "answer_selectivity_all_features": 0.0059,
  "added_over_baseline_ratio": 3.88,
  "top_added_feature_ids": [
    15402,
    1891,
    9432,
    8581,
    2045,
    11527,
    159,
    14885,
    629,
    15798,
    12748,
    13487,
    8116,
    5816,
    8591
  ],
  "top_removed_feature_ids": [
    6546,
    8099,
    1530,
    16128,
    907,
    2560,
    14971,
    13488,
    13460,
    8438,
    9802,
    8828,
    238,
    13716,
   

In [34]:
# === CELL 18 (EXP11): DEPTH DOSE-RESPONSE — donor solve rate, presence, conferral per depth ===
# Base: CELL E11 of Gemma_27Bto2B_eval_CORE.ipynb. ONE task family (chained integer ops); only
# the NUMBER OF SEQUENTIAL STEPS varies. Magnitude is held roughly flat on purpose so the bins
# don't confound "deeper" with "bigger numbers".
#   s1: a*b     s2: a*b+c     s3: a*b+c+d     s4: a*b+c+d+e
# Reports per depth: DONOR SOLVE RATE (P(donor solves in one pass)), DONOR PRESENCE (digit-1
# probe at the graft site) and CONFERRAL — the three terms of the law, as a function of depth.
# Donor forwards use DONOR_BATCH, and the generators use fixed seeds, so the donor columns are
# again directly comparable between the 9B- and 2B-recipient notebooks.
RUN_DOSE = globals().get("RUN_DOSE", True)
if RUN_DOSE:
    def _depth_expr(rng, steps):
        a, b = rng.randint(12, 49), rng.randint(12, 49)        # 2-digit x 2-digit: hard for small recipients
        val = a * b; expr = f"{a} * {b}"
        if steps >= 2:
            c = rng.randint(2, 30); val = val + c; expr = f"{expr} + {c}"
        if steps >= 3:
            d = rng.randint(2, 30); val = val + d; expr = f"{expr} + {d}"   # add: keeps magnitude flat
        if steps >= 4:
            e = rng.randint(2, 30); val = val + e; expr = f"{expr} + {e}"
        return expr, val
    def gen_depth_bin(n, rng, steps, exclude=None):
        exclude = exclude or set(); out, seen = [], set(); tries = 0
        while len(out) < n and tries < n * 400:
            tries += 1
            expr, ans = _depth_expr(rng, steps)
            if expr in seen or expr in exclude: continue
            seen.add(expr)
            ids, tok_id = _aencode(tokenizer, expr, ans)
            if tok_id is None: continue
            out.append(dict(expr=expr, ans=ans, ids=ids, tok=tok_id))
        return out
    DOSE_TRAIN = {1: 800, 2: min(N_ARITH_TRAIN, 1500), 3: min(N_ARITH_TRAIN, 1500), 4: min(N_ARITH_TRAIN, 1500)}
    DOSE_EVAL  = {1: 500, 2: N_DOSE, 3: N_DOSE, 4: N_DOSE}
    dose = {}
    for steps in DOSE_STEPS:
        name = f"steps{steps}"
        trn = gen_depth_bin(DOSE_TRAIN[steps], random.Random(70 + steps), steps)
        evl = gen_depth_bin(DOSE_EVAL[steps], random.Random(80 + steps), steps, {p["expr"] for p in trn})
        if len(trn) < 50 or len(evl) < 50:
            dose[name] = {"note": f"insufficient unique problems (train={len(trn)}, eval={len(evl)})"}
            print(f"  [{name}] skipped — only train={len(trn)}, eval={len(evl)}"); continue
        X9tr, _ = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in trn], batch=DONOR_BATCH); X9tr = X9tr[L9_SINGLE]
        X9ev, ntop9 = states_and_top(model_9b, [L9_SINGLE], [p["ids"] for p in evl], batch=DONOR_BATCH); X9ev = X9ev[L9_SINGLE]
        donor_solves = float(np.mean([ntop9[i] == evl[i]["tok"] for i in range(len(evl))]))
        keepd = [i for i in range(len(evl)) if ntop9[i] == evl[i]["tok"]]
        evl_k = [evl[i] for i in keepd]; X9ev_k = X9ev[keepd]
        if len(evl_k) < 20:
            dose[name] = {"donor_solve_rate": round(donor_solves, 3), "n_donor_solved": len(evl_k),
                          "note": "donor solves too few — answer not linearly present at this depth"}
            print(f"  [{name}] donor_solve_rate={donor_solves:.2f}; donor-solved n={len(evl_k)} (answer absent)")
            continue
        _, ntop2 = states_and_top(model_2b, [L2_SINGLE], [p["ids"] for p in evl_k])
        nat = float(np.mean([ntop2[i] == evl_k[i]["tok"] for i in range(len(evl_k))]))
        yb = torch.tensor([fd(p["ans"]) for p in evl_k])
        probe = float(np.mean(probe_split_bools(X9ev_k, yb)))
        torch.manual_seed(0)
        Wb = Wr.clone().to(DEVICE).requires_grad_(True); bb = mu2.clone().to(DEVICE).requires_grad_(True)
        optb = torch.optim.Adam([Wb, bb], lr=1e-3)
        model_2b.requires_grad_(False)
        hwb = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch)
        ii = list(range(len(trn)))
        try:
            for ep in range(TASK_EPOCHS):
                random.Random(ep).shuffle(ii)
                for s in range(0, len(ii), ARITH_BATCH):
                    sub = ii[s:s+ARITH_BATCH]
                    _graft["vec"] = (X9tr[sub].to(DEVICE) - mu9d) @ Wb + bb
                    ids, m = left_pad([trn[k]["ids"] for k in sub], tokenizer.pad_token_id)
                    lg = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].float()
                    tgt = torch.tensor([trn[k]["tok"] for k in sub], device=DEVICE)
                    lb = F.cross_entropy(lg, tgt); optb.zero_grad(); lb.backward(); optb.step()
        finally:
            hwb.remove(); _graft["vec"] = None
        model_2b.requires_grad_(True)
        Wb, bb = Wb.detach(), bb.detach()
        uns = [i for i in range(len(evl_k)) if ntop2[i] != evl_k[i]["tok"]]
        @torch.inference_mode()
        def _confer_bin(idxs, X9ev_k=X9ev_k, evl_k=evl_k, Wb=Wb, bb=bb):
            h = model_2b.model.layers[L2_SINGLE].register_forward_hook(patch_vec_batch); out = []
            try:
                for i in range(0, len(idxs), ARITH_BATCH):
                    sub = idxs[i:i+ARITH_BATCH]
                    _graft["vec"] = (X9ev_k[sub].to(DEVICE) - mu9d) @ Wb + bb
                    ids, m = left_pad([evl_k[j]["ids"] for j in sub], tokenizer.pad_token_id)
                    top = model_2b(ids.to(DEVICE), attention_mask=m.to(DEVICE)).logits[:, -1, :].argmax(-1).cpu().tolist()
                    out += [top[k] == evl_k[sub[k]]["tok"] for k in range(len(sub))]
            finally:
                h.remove(); _graft["vec"] = None
            return out
        confer = fmt(wilson_bools(_confer_bin(uns))) if uns else "n/a (recipient solves all)"
        dose[name] = {"n_eval_donor_solved": len(evl_k), "n_unsolv_by_recipient": len(uns),
                      "donor_solve_rate": round(donor_solves, 3),
                      "native_recipient_first": round(nat, 3),
                      "donor_presence_digit1_probe": round(probe, 3),
                      "task_confer_unsolv_first": confer,
                      "example": evl_k[0]["expr"]}
        print(f"  [{name}] donor_solve={donor_solves:.2f} presence={probe:.2f} "
              f"native={nat:.2f} confer={confer} (unsolv n={len(uns)}) eg: {evl_k[0]['expr']}", flush=True)
    RESULTS["dose_response_depth"] = dose
    print("EXP11 depth dose-response:", json.dumps(dose, indent=2))
else:
    print("EXP11 dose-response skipped (RUN_DOSE=False).")


EXP11 dose-response skipped (RUN_DOSE=False).


In [35]:
# === CELL 19 (EXP5): GSM8K SINGLE-SITE STITCH — boundary check on ~400 problems ===
# Base: CELL 12 of consolidated_eval.ipynb, cut to the SINGLE-SITE condition and SUBSAMPLED to
# GSM_EVAL problems (a full 1319-problem sweep is not affordable with a 27B donor in the loop).
# GSM8K is the "answer ABSENT at the graft site" side of the contrast: the donor cannot solve it
# in one prefill pass, so by the law there is nothing for the stitch to transcribe and conferral
# should stay at the recipient's baseline no matter how good the bridge is.
RUN_GSM = globals().get("RUN_GSM", True)
if RUN_GSM:
    def gsm_collect(rows, batch=GSM_BATCH, maxtok=GSM_FIT_TOKENS):
        X9, X2, ntok = [], [], 0
        for i in range(0, len(rows), batch):
            enc = tokenizer([gsm_prompt(r["question"]) for r in rows[i:i+batch]],
                            return_tensors="pt", padding=True, truncation=True, max_length=512).to(DEVICE)
            msk = enc["attention_mask"].bool()
            with torch.inference_mode():
                h9 = model_9b.model(enc["input_ids"], attention_mask=enc["attention_mask"],
                                    output_hidden_states=True).hidden_states[L9_SINGLE+1]
                h2 = model_2b.model(enc["input_ids"], attention_mask=enc["attention_mask"],
                                    output_hidden_states=True).hidden_states[L2_SINGLE+1]
            X9.append(h9.float()[msk].cpu()); X2.append(h2.float()[msk].cpu())
            ntok += int(msk.sum())
            if ntok >= maxtok: break
        return torch.cat(X9), torch.cat(X2)
    gfit = list(load_dataset("openai/gsm8k","main",split="train").select(range(GSM_FIT)))
    X9g, X2g = gsm_collect(gfit)
    print(f"  GSM8K fit states: {X9g.shape[0]} token positions")
    stitch = fit_ridge(X9g, X2g)
    stitch = (stitch[0].to(DEVICE), stitch[1].to(DEVICE), stitch[2].to(DEVICE))
    # held-out recon_cos (refit on 80%, score on 20%)
    cut = int(0.8 * X9g.shape[0])
    mh = fit_ridge(X9g[:cut], X2g[:cut]); mh = (mh[0].to(DEVICE), mh[1].to(DEVICE), mh[2].to(DEVICE))
    rch = F.cosine_similarity(apply_map(X9g[cut:].to(DEVICE), mh).cpu(), X2g[cut:], dim=1).numpy()
    recon_cos_gsm = fmt(bootstrap_ci(rch))
    del X9g, X2g

    gctx = {"strength": 0.0, "grafted": None}
    def gsm_hook(m, _i, o):
        h = _hid(o); s = gctx["strength"]; gd = gctx["grafted"]
        if s == 0 or gd is None or h.shape[1] != gd.shape[1]: return o
        td = next(m.parameters()).dtype
        return _pack(o, ((1-s)*h.float() + s*gd.float().to(h.device)).to(td))
    @torch.inference_mode()
    def gsm_eval(model, rows, strength, batch=GSM_BATCH):
        handle = None
        if model is model_2b:
            model_2b.model.layers[L2_SINGLE]._forward_hooks.clear()
            handle = model_2b.model.layers[L2_SINGLE].register_forward_hook(gsm_hook)
        gctx["strength"] = strength if model is model_2b else 0.0; ok = []
        try:
            for i in range(0, len(rows), batch):
                b = rows[i:i+batch]
                enc = tokenizer([gsm_prompt(r["question"]) for r in b], return_tensors="pt",
                                padding=True, truncation=True, max_length=512).to(DEVICE)
                if gctx["strength"] > 0:
                    ob = model_9b.model(enc["input_ids"], attention_mask=enc["attention_mask"],
                                        output_hidden_states=True).hidden_states[L9_SINGLE+1]
                    gctx["grafted"] = apply_map(ob, stitch)
                else:
                    gctx["grafted"] = None
                gen = model.generate(**enc, max_new_tokens=MAX_NEW_GSM, do_sample=False,
                                     pad_token_id=tokenizer.eos_token_id)
                txt = tokenizer.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
                for r, t in zip(b, txt):
                    p, g = gsm_extract(t), gsm_extract(r["answer"])
                    ok.append(p is not None and g is not None and abs(p-g) < 0.01)
                if i % (batch*10) == 0: print(f"    gsm {i}/{len(rows)}", flush=True)
        finally:
            if handle: handle.remove()
            gctx.update(strength=0.0, grafted=None)
        return ok
    geval = list(load_dataset("openai/gsm8k","main",split="test").select(range(GSM_EVAL)))
    gres = {"n_eval": len(geval), "n_fit_rows": GSM_FIT, "recon_cos_heldout": recon_cos_gsm,
            "recipient_baseline": fmt(wilson_bools(gsm_eval(model_2b, geval, 0.0)))}
    for s in GSM_STRENGTHS:
        gres[f"single_site_s{s}"] = fmt(wilson_bools(gsm_eval(model_2b, geval, s)))
    if RUN_GSM_DONOR:
        # DONOR-SIDE quantity: P(donor solves) on GSM8K. Identical model in both notebooks, so
        # this may be run once and copied — it is kept here so each JSON is self-contained.
        gdon = geval[:GSM_DONOR_N]
        gres["donor_solve_rate"] = fmt(wilson_bools(gsm_eval(model_9b, gdon, 0.0)))
        gres["n_donor_eval"] = len(gdon)
    RESULTS["gsm8k_stitch"] = gres
    print("EXP5 GSM8K single-site stitch:", json.dumps(gres, indent=2))
else:
    print("EXP5 GSM8K skipped (RUN_GSM=False).")


EXP5 GSM8K skipped (RUN_GSM=False).


In [36]:
# === CELL 21: SAVE — concentrated results table + JSON ===
RESULTS["config"] = {
    "donor": MODEL_9B, "recipient": MODEL_2B,
    "single_pair_recipient_donor": [L2_SINGLE, L9_SINGLE],
    "layer_pairs": [list(p) for p in LAYER_PAIRS],
    "d_recipient": D_RECIP, "d_donor": D_DONOR,
    "layers_recipient": NL_RECIP, "layers_donor": NL_DONOR,
    "n_arith_train": N_ARITH_TRAIN, "n_arith_eval": N_ARITH_EVAL,
    "arith_batch": ARITH_BATCH, "gsm_batch": GSM_BATCH, "donor_batch": DONOR_BATCH,
    "task_seeds": TASK_SEEDS, "task_epochs": TASK_EPOCHS,
    "gsm_eval_n": GSM_EVAL, "ceil_n": CEIL_N, "smoke_test": SMOKE_TEST,
    "donor_quantized": QUANTIZE_9B,
    "arm": ARM, "use_chat_template": USE_CHAT_TEMPLATE,
}
print("="*78); print("ALL RESULTS (point [95% CI])"); print("="*78)
print(json.dumps(RESULTS, indent=2))

print("\n" + "="*78)
print("HEADLINE NUMBERS FOR THE DONOR-HELD-FIXED COMPARISON")
print("="*78)
_a = RESULTS.get("arithmetic", {}); _e = RESULTS.get("answer_subspace_erasure", {})
_r = RESULTS.get("recipient_feature_delta", {}); _b = RESULTS.get("bins", {})
rows = [
    ("recipient",                          MODEL_2B),
    ("recipient d_model",                  D_RECIP),
    ("DONOR presence (digit-1 @ graft)",   RESULTS.get("donor_presence", {}).get(
                                               "arith_donor_probe", "not run (RUN_PRESENCE=False)")),
    ("DONOR solve rate (arith)",           _b.get("donor_solve_rate")),
    ("unsolvable bin n",                   RESULTS.get("bins", {}).get("n_unsolvable_bin")),
    ("conferral first-token (task)",       _a.get("task_unsolv_first_pooled")),
    ("conferral full-answer (5 seeds)",    _a.get("task_unsolv_full_acrossseed")),
    ("native full-answer (floor)",         _a.get("native_unsolv_full")),
    ("shuffle control (first)",            _a.get("shuffle_unsolv_first")),
    ("free-vector ceiling (full)",         RESULTS.get("freevec_ceiling", {}).get("ceiling_full")),
    ("erasure collapse rank",              _e.get("collapse_rank")),
    ("erasure collapse rank / d_model",    _e.get("collapse_rank_over_dmodel")),
    ("SAE added-feature sel. ratio",       _r.get("added_over_baseline_ratio")),
    ("SAE total feature movement (L1)",    _r.get("task_stitch_L1_change")),
    ("GSM8K baseline",                     RESULTS.get("gsm8k_stitch", {}).get("recipient_baseline")),
    ("GSM8K single-site stitch",           RESULTS.get("gsm8k_stitch", {}).get("single_site_s1.0")),
]
for k, v in rows: print(f"  {k:36s} {v}")

with open(OUT_JSON, "w") as f: json.dump(RESULTS, f, indent=2)
print(f"\nsaved {OUT_JSON}")
print("NOTE: the donor_side join-key block is not built in this notebook (it was for the "
      "donor-held-fixed 9B-vs-2B-recipient comparison). The two ARMS here use DIFFERENT donors, "
      "so join the two arm JSONs on results['config']['arm'] instead.")


ALL RESULTS (point [95% CI])
{
  "patching": {
    "n_used_recipient": 150,
    "n_used_donor": 150,
    "causal_peak_recipient": 23,
    "causal_peak_donor": 39,
    "graft_layer_recipient": 20,
    "recovery_recipient_at_graft": "0.748 [0.738, 0.757]",
    "best_donor_match_for_graft": 34,
    "cka_at_graft": 0.944,
    "cka_at_configured_pair": 0.944,
    "recovery_curve_recipient": [
      -0.0,
      0.0,
      -0.0,
      -0.0,
      -0.0,
      0.0,
      -0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.01,
      0.01,
      0.07,
      0.23,
      0.25,
      0.72,
      0.72,
      0.75,
      0.76,
      0.76,
      0.78,
      0.8,
      1.0
    ],
    "recovery_curve_donor": [
      -0.0,
      0.0,
      0.0,
      0.0,
      -0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
      0.0,
     